In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

# Import pyDTDM modules
from pyDTDM.feature_engineering import GeologyAwareFeatureSelector, identify_feature_groups
from pyDTDM.pu_learning import PUBaggingClassifier, validate_pu_data
from pyDTDM.pu_diagnostics import PUFeatureDiagnostics


# North American Copper Prospectivity Mapping
## PU-Learning Workflow with Geology-Aware Feature Engineering

---

### Overview

This notebook implements a **publication-ready** Positive-Unlabeled (PU) learning workflow for copper prospectivity mapping in North America. The approach properly handles label uncertainty where:

- **Positive samples** = Known copper deposits (reliable)
- **Unlabeled samples** = Background locations (NOT true negatives - may contain undiscovered deposits)

### Key Features

1. **Geology-Aware Feature Preprocessing**
   - Correlation-based near-duplicate removal
   - Variance filtering
   - Geological group preservation
   - NO destructive transformations (PCA avoided)

2. **PU-Bagging Classifier**
   - All positive samples in every bag
   - Bootstrap sampling of unlabeled samples only
   - Strict regularization for spatial stability
   - Uncertainty quantification via prediction variance

3. **Post-PU Diagnostics**
   - SHAP-based feature interpretability
   - Feature stability analysis
   - Automatic unstable feature identification
   - Geological group-level importance

4. **Tonnage Weighting**
   - Sample weights from deposit tonnage (tonnage_mt)
   - Accounts for deposit size during training
   - Handles missing values (-9999 → NaN)

### Workflow Steps

1. Load and prepare real copper prospectivity data
2. Apply geology-aware feature hygiene
3. Train-test split with stratification
4. Train PU-bagging classifier with tonnage weights
5. Predict with uncertainty quantification
6. Analyze feature importances
7. Perform SHAP-based diagnostics
8. Generate spatial prospectivity maps

### Expected Outputs

- **CSVs**: Feature removal report, importance rankings, stability analysis, high-confidence targets
- **Plots**: Predictions, importance, stability diagnostics, spatial maps
- **Performance**: AUC-ROC, AUC-PR, uncertainty metrics

### Requirements

```python
numpy, pandas, scikit-learn, matplotlib, seaborn
shap (optional, for interpretability)
```

### Citation

If using this workflow, cite:
- pyDTDM PU-Learning implementation
- Elkan & Noto (2008) - PU learning foundation
- Lundberg & Lee (2017) - SHAP interpretability



In [ ]:
# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("PU-LEARNING WORKFLOW FOR MINERAL PROSPECTIVITY MAPPING")
print("="*80)
print(f"Random seed: {RANDOM_STATE}")
print(f"Purpose: Scientific publication with full reproducibility")
print("="*80 + "\n")



In [ ]:
# ==============================================================================
# STEP 1: LOAD AND PREPARE REAL MINERAL PROSPECTIVITY DATA
# ==============================================================================
print("\n" + "="*80)
print("STEP 1: Load Real North American Copper Prospectivity Dataset")
print("="*80)

# Load the data
data_path = "<DATA_ROOT>/Paper/Zenodo_DataBundle/training_data/spatial_training_data_shuffled.csv"
ALL_DATA = pd.read_csv(data_path)

print(f"Original dataset shape: {ALL_DATA.shape}")
print(f"Total columns: {len(ALL_DATA.columns)}")

# Remove unnecessary columns
columns_to_remove = ['Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1']
ALL_DATA = ALL_DATA.drop(columns=[col for col in columns_to_remove if col in ALL_DATA.columns])

print(f"\n✓ Removed {len([c for c in columns_to_remove if c in ALL_DATA.columns])} unnamed columns")
print(f"Dataset shape after cleanup: {ALL_DATA.shape}")

# Prepare weights from tonnage_mt
# Handle -9999 as NaN
ALL_DATA['tonnage_mt'] = ALL_DATA['tonnage_mt'].replace(-9999, np.nan)

# Analyze tonnage distribution
positive_mask = ALL_DATA['label_binary'] == 1
positive_with_tonnage = positive_mask & ALL_DATA['tonnage_mt'].notna()

if positive_with_tonnage.sum() > 0:
    tonnage_values = ALL_DATA.loc[positive_with_tonnage, 'tonnage_mt']
    
    print(f"\nTonnage distribution (deposits with known tonnage):")
    print(f"  Count: {len(tonnage_values)}")
    print(f"  Min: {tonnage_values.min():.2f} Mt")
    print(f"  25th percentile: {tonnage_values.quantile(0.25):.2f} Mt")
    print(f"  Median: {tonnage_values.median():.2f} Mt")
    print(f"  75th percentile: {tonnage_values.quantile(0.75):.2f} Mt")
    print(f"  Max: {tonnage_values.max():.2f} Mt")
    print(f"  Mean: {tonnage_values.mean():.2f} Mt")
    print(f"  Std: {tonnage_values.std():.2f} Mt")
    
    # Plot tonnage distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Original scale
    ax = axes[0]
    ax.hist(tonnage_values, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_xlabel('Tonnage (Mt)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Tonnage Distribution (Original Scale)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Log scale
    ax = axes[1]
    ax.hist(np.log1p(tonnage_values), bins=30, edgecolor='black', alpha=0.7, color='coral')
    ax.set_xlabel('Log(1 + Tonnage)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Tonnage Distribution (Log Scale)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('tonnage_distribution.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Tonnage distribution plot saved to: tonnage_distribution.png")
    plt.show()

# Create weights: use log(1+tonnage) for positives to handle skewed distribution
ALL_DATA['sample_weight'] = 1.0  # Default weight for unlabeled

if positive_with_tonnage.sum() > 0:
    # Use log transformation to reduce impact of extreme values - use directly
    log_weights = ALL_DATA.loc[positive_with_tonnage, 'tonnage_mt']
    ALL_DATA.loc[positive_with_tonnage, 'sample_weight'] = log_weights
    
    print(f"\nWeight statistics (log-transformed, used directly):")
    print(f"  Positive samples with tonnage: {positive_with_tonnage.sum()}")
    print(f"  Positive samples without tonnage: {(positive_mask & ALL_DATA['tonnage_mt'].isna()).sum()}")
    print(f"  Min weight (positive w/ tonnage): {log_weights.min():.3f}")
    print(f"  Median weight (positive w/ tonnage): {log_weights.median():.3f}")
    print(f"  Mean weight (positive w/ tonnage): {log_weights.mean():.3f}")
    print(f"  Max weight (positive w/ tonnage): {log_weights.max():.3f}")
    print(f"  Weight formula: log(1 + tonnage_mt)")
    print(f"  Note: XGBoost/sklearn handle weights directly - no normalization needed")
else:
    print(f"\nNo tonnage data available - using uniform weights")

# Separate features, target, and metadata
metadata_cols = ['Latitude', 'Longitude', 'label_binary', 'tonnage_mt', 'sample_weight', 
                 'weights', 'SHAPE_TYPE', 'Shape_Leng', 'Litho_Group']

feature_cols = [col for col in ALL_DATA.columns if col not in metadata_cols]

X_df = ALL_DATA[feature_cols].copy()
y = ALL_DATA['label_binary'].values
sample_weights = ALL_DATA['sample_weight'].values

print(f"\nFeature extraction:")
print(f"  Feature columns: {len(feature_cols)}")
print(f"  Metadata columns: {len(metadata_cols)}")

# Validate PU data
print(f"\nDataset characteristics:")
print(f"  Total samples: {len(y):,}")
print(f"  Positive samples (known deposits): {np.sum(y == 1):,}")
print(f"  Unlabeled samples: {np.sum(y == 0):,}")
print(f"  Positive rate: {np.sum(y == 1)/len(y):.2%}")

# Validate PU data
diagnostics = validate_pu_data(y, verbose=True)

In [ ]:
list(ALL_DATA.columns)

In [ ]:
# ==============================================================================
# STEP 1.5: DEFINE GEOLOGICAL FEATURE GROUPS (MINERAL SYSTEMS FRAMEWORK)
# ==============================================================================
print("\n" + "="*80)
print("STEP 1.5: Define Geological Feature Groups for Interpretability")
print("="*80)

# Define feature groups using simplified patterns
# Order matters: more specific patterns first
feature_groups_patterns = {
    'PROXIMITY': [
        '*_nearest_distance*'
    ],
    'STRUCTURAL': [
        # Gradient-based structural indicators
        '*_x_grad*',
        '*_y_grad*',
        '*_magnitude_grad*'
    ],
    'TEXTURE': [
        # GLCM texture features
        '*contrast*',
        '*dissimilarity*',
        '*homogeneity*',
        '*energy*',
        '*correlation*',
        '*ASM*'
    ],
    'ALTERATION': [
        # Geochemical/spectral alteration signatures
        'al2o3*',
        'hydrothermal*',
        'sio2*',
        'ferrous*',
        'ferric*'
    ],
    'FERTILITY': [
        # Geophysical fertility indicators
        'Geophysics*',
        # 'Litho*'
    ]
}

print("\nFeature groups defined:")
for group in feature_groups_patterns.keys():
    print(f"  - {group}")

# Count features per group using proper pattern matching
import re
feature_group_counts = {}

for group_name, patterns in feature_groups_patterns.items():
    matched_cols = set()
    for pattern in patterns:
        # Convert glob pattern to regex
        regex_pattern = pattern.replace('*', '.*')
        for col in X_df.columns:
            if re.match(f'^{regex_pattern}$', col, re.IGNORECASE):
                matched_cols.add(col)
    
    feature_group_counts[group_name] = len(matched_cols)

print("\nFeature counts per group:")
for group, count in sorted(feature_group_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {group}: {count} features")

print(f"\nTotal features to process: {X_df.shape[1]}")
print("="*80)

In [ ]:
feature_groups_patterns

In [ ]:
# ==============================================================================
# STEP 2: GEOLOGY-AWARE FEATURE PREPROCESSING
# ==============================================================================
print("\n" + "="*80)
print("STEP 2: Geology-Aware Feature Hygiene")
print("="*80)

# Use the same feature_groups_patterns for the selector
print("\nApplying geology-aware feature selection...")

# Initialize feature selector
feature_selector = GeologyAwareFeatureSelector(
    correlation_threshold=0.95,
    variance_threshold=0.001,
    # feature_groups=feature_groups,
    selection_strategy='median_corr',
    random_state=RANDOM_STATE,
    verbose=True,
    feature_groups=feature_groups_patterns)

# Fit and transform
X_clean = feature_selector.fit_transform(X_df)

print(f"\n✓ Feature preprocessing complete")
print(f"  Original features: {X_df.shape[1]}")
print(f"  Retained features: {X_clean.shape[1]}")
print(f"  Reduction: {100*(1 - X_clean.shape[1]/X_df.shape[1]):.1f}%")

# Get removal report
removal_report = feature_selector.get_removal_report()
print(f"\nFeatures removed by reason:")
print(removal_report['reason'].value_counts())

# Display correlation clusters (top 5)
if len(feature_selector.correlation_clusters_) > 0:
    cluster_report = feature_selector.get_cluster_report()
    print(f"\nFound {len(feature_selector.correlation_clusters_)} correlation clusters")
    print(f"\nTop 5 largest correlation clusters:")
    top_clusters = cluster_report.groupby('cluster_id')['n_cluster_size'].first().nlargest(5)
    for cluster_id in top_clusters.index:
        cluster_data = cluster_report[cluster_report['cluster_id'] == cluster_id]
        print(f"\n  Cluster {cluster_id} ({len(cluster_data)} features):")
        print(f"    Representative: {cluster_data[cluster_data['is_representative']]['feature'].values[0]}")
        removed = cluster_data[~cluster_data['is_representative']]['feature'].values[:3]
        if len(removed) > 0:
            print(f"    Removed: {', '.join(removed)}{'...' if len(cluster_data) > 4 else ''}")

# Save removal report
removal_report.to_csv('feature_removal_report.csv', index=False)
print(f"\n✓ Feature removal report saved to: feature_removal_report.csv")

In [ ]:
# list(X_train.columns)

In [ ]:
# ==============================================================================
# STEP 3: TRAIN-TEST SPLIT WITH STRATIFICATION
# ==============================================================================
print("\n" + "="*80)
print("STEP 3: Train-Test Split with Stratification")
print("="*80)

# Stratified split to maintain positive rate
X_train, X_test, y_train, y_test, weights_train, weights_test = train_test_split(
    X_clean, y, sample_weights,
    test_size=0.3, 
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"  Positive (deposits): {np.sum(y_train==1):,}")
print(f"  Unlabeled: {np.sum(y_train==0):,}")
print(f"  Positive rate: {np.sum(y_train==1)/len(y_train):.2%}")

print(f"\nTest set: {X_test.shape[0]:,} samples")
print(f"  Positive (deposits): {np.sum(y_test==1):,}")
print(f"  Unlabeled: {np.sum(y_test==0):,}")
print(f"  Positive rate: {np.sum(y_test==1)/len(y_test):.2%}")

print(f"\nWeight statistics (training set):")
print(f"  Mean weight (positive): {weights_train[y_train==1].mean():.2f}")
print(f"  Median weight (positive): {np.median(weights_train[y_train==1]):.2f}")
print(f"  Max weight (positive): {weights_train[y_train==1].max():.2f}")
print(f"  Min weight (positive): {weights_train[y_train==1].min():.2f}")

In [ ]:
# ==============================================================================
# STEP 3.1: OPTIONAL - DOWNSAMPLE NEGATIVE (UNLABELED) SAMPLES
# ==============================================================================
print("\n" + "="*80)
print("STEP 3.1: Downsample Negative Samples (Optional)")
print("="*80)

# Configuration: Set to None to keep all samples, or a float between 0 and 1 to drop that fraction
# Examples: 0.4 = drop 40%, 0.5 = drop 50% of negative samples
DROP_NEGATIVE_FRACTION = 0.5  # Set to 0.4, 0.5, or None

if DROP_NEGATIVE_FRACTION is not None and 0 < DROP_NEGATIVE_FRACTION < 1:
    print(f"\n⚙ Downsampling enabled: Dropping {DROP_NEGATIVE_FRACTION*100:.0f}% of negative samples")
    
    # Get indices of positive and negative samples in training set
    positive_train_idx = np.where(y_train == 1)[0]
    negative_train_idx = np.where(y_train == 0)[0]
    
    # Calculate how many negative samples to keep
    n_negative_keep = int(len(negative_train_idx) * (1 - DROP_NEGATIVE_FRACTION))
    
    print(f"\nTraining set before downsampling:")
    print(f"  Positive samples: {len(positive_train_idx):,}")
    print(f"  Negative samples: {len(negative_train_idx):,}")
    print(f"  Positive ratio: {len(positive_train_idx)/len(y_train)*100:.2f}%")
    print(f"  Total: {len(y_train):,}")
    
    # Randomly sample negative indices to keep
    np.random.seed(RANDOM_STATE)
    negative_keep_idx = np.random.choice(negative_train_idx, size=n_negative_keep, replace=False)
    
    # Combine positive and sampled negative indices
    keep_train_idx = np.concatenate([positive_train_idx, negative_keep_idx])
    np.random.shuffle(keep_train_idx)  # Shuffle to mix positive and negative
    
    # Update training data
    X_train = X_train.iloc[keep_train_idx].reset_index(drop=True)
    y_train = y_train[keep_train_idx]
    weights_train = weights_train[keep_train_idx]
    
    print(f"\nTraining set after downsampling:")
    print(f"  Positive samples: {np.sum(y_train==1):,}")
    print(f"  Negative samples: {np.sum(y_train==0):,}")
    print(f"  Positive ratio: {np.sum(y_train==1)/len(y_train)*100:.2f}%")
    print(f"  Total: {len(y_train):,}")
    print(f"  Reduction: {len(negative_train_idx) - np.sum(y_train==0):,} negative samples removed")
    
    # Note: Test set remains unchanged
    print(f"\n✓ Training set downsampled successfully")
    print(f"  Test set unchanged: {len(y_test):,} samples")
    
else:
    print(f"\n⚙ Downsampling disabled (DROP_NEGATIVE_FRACTION = {DROP_NEGATIVE_FRACTION})")
    print(f"  Using all training samples: {len(y_train):,}")
    print(f"  To enable, set DROP_NEGATIVE_FRACTION to a value between 0 and 1")
    print(f"  Example: DROP_NEGATIVE_FRACTION = 0.4  # Drop 40% of negatives")

print("\n" + "="*80)

In [ ]:
# ==============================================================================
# STEP 3.5: HYPERPARAMETER TUNING WITH RANDOMIZED SEARCH CV
# ==============================================================================
print("\n" + "="*80)
print("STEP 3.5: Hyperparameter Tuning for XGBoost Base Estimator")
print("="*80)

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import uniform, randint

# Define parameter distributions for RandomizedSearchCV
param_distributions = {
    'max_depth': randint(3, 12),              # Integer range: 3-11
    'learning_rate': uniform(0.01, 0.29),     # Continuous: 0.01-0.30
    'min_child_weight': randint(1, 7),        # Integer range: 1-6
    'subsample': uniform(0.6, 0.4),           # Continuous: 0.6-1.0
    'colsample_bytree': uniform(0.6, 0.4),    # Continuous: 0.6-1.0
    'reg_alpha': uniform(0, 1.0),             # Continuous: 0.0-1.0
    'reg_lambda': uniform(0.1, 2.9)           # Continuous: 0.1-3.0
}

print("\nParameter search space:")
print("  max_depth: 3-11 (integer)")
print("  learning_rate: 0.01-0.30 (continuous)")
print("  min_child_weight: 1-6 (integer)")
print("  subsample: 0.6-1.0 (continuous)")
print("  colsample_bytree: 0.6-1.0 (continuous)")
print("  reg_alpha: 0.0-1.0 (continuous)")
print("  reg_lambda: 0.1-3.0 (continuous)")

n_iter = 100  # Number of random combinations to try
print(f"\nRandomized search will sample {n_iter} parameter combinations")
print("Note: This samples the continuous space efficiently for publication quality")

# Base XGBoost classifier for tuning
base_xgb = xgb.XGBClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='logloss',
    use_label_encoder=False
)

# Setup stratified k-fold for PU learning
# Important: Stratify to maintain positive/unlabeled ratio
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"\nCross-validation strategy: 5-fold Stratified K-Fold")
print(f"  This ensures positive/unlabeled ratio is maintained in each fold")

# Randomized search with sample weights
print("\nRunning Randomized Search CV...")
print(f"This may take 15-45 minutes depending on dataset size...")
print("Progress will be displayed as iterations complete\n")

random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring='roc_auc',        # Use AUC-ROC for PU learning
    n_jobs=-1,                # Parallelize across CV folds
    verbose=2,
    refit=True,
    random_state=RANDOM_STATE,
    return_train_score=True   # Track overfitting
)

# Fit with sample weights
random_search.fit(X_train, y_train, sample_weight=weights_train)

print(f"\n✓ Randomized Search complete")

# Report best parameters
print("\n" + "="*80)
print("BEST HYPERPARAMETERS FOUND:")
print("="*80)
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV AUC-ROC: {random_search.best_score_:.4f}")

# Show top 10 parameter combinations
cv_results = pd.DataFrame(random_search.cv_results_)
top_10 = cv_results.nlargest(10, 'mean_test_score')[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
]

print("\nTop 10 parameter combinations:")
for idx, row in top_10.iterrows():
    print(f"\n  Rank {int(row['rank_test_score'])}:")
    print(f"    Score: {row['mean_test_score']:.4f} (+/- {row['std_test_score']:.4f})")
    print(f"    Parameters: {row['params']}")

# Save CV results
cv_results.to_csv('hyperparameter_tuning_results.csv', index=False)
print(f"\n✓ Full CV results saved to: hyperparameter_tuning_results.csv")

# Analyze parameter importance via correlation
print("\n" + "="*80)
print("PARAMETER SENSITIVITY ANALYSIS")
print("="*80)

param_correlations = {}
print("\nParameter impact on performance (correlation with AUC-ROC):")
for param in param_distributions.keys():
    param_col = f'param_{param}'
    if param_col in cv_results.columns:
        # Convert to numeric
        param_values = pd.to_numeric(cv_results[param_col], errors='coerce')
        if not param_values.isna().all():
            corr = param_values.corr(cv_results['mean_test_score'])
            param_correlations[param] = corr
            impact_str = '(+increase helps)' if corr > 0.1 else '(-decrease helps)' if corr < -0.1 else '(minimal impact)'
            print(f"  {param}: {corr:+.3f} {impact_str}")

# Visualize hyperparameter tuning results
print("\nGenerating hyperparameter tuning visualizations...")

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# Plot 1: Score distribution
ax = axes[0]
ax.hist(cv_results['mean_test_score'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
ax.axvline(random_search.best_score_, color='red', linestyle='--', linewidth=2, 
           label=f'Best: {random_search.best_score_:.4f}')
ax.set_xlabel('Cross-Validation AUC-ROC', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of CV Scores', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Score vs iteration
ax = axes[1]
iterations = range(len(cv_results))
ax.plot(iterations, cv_results['mean_test_score'], 'o-', alpha=0.6, markersize=4)
ax.axhline(random_search.best_score_, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Cross-Validation AUC-ROC', fontsize=11)
ax.set_title('Search Progress', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Parameter sensitivity (correlations)
ax = axes[2]
if param_correlations:
    sorted_params = sorted(param_correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    param_names = [p[0] for p in sorted_params]
    param_corrs = [p[1] for p in sorted_params]
    colors = ['green' if c > 0 else 'red' for c in param_corrs]
    
    y_pos = np.arange(len(param_names))
    ax.barh(y_pos, param_corrs, color=colors, alpha=0.7, edgecolor='black')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(param_names, fontsize=10)
    ax.set_xlabel('Correlation with AUC-ROC', fontsize=11)
    ax.set_title('Parameter Sensitivity', fontsize=12, fontweight='bold')
    ax.axvline(0, color='black', linewidth=1)
    ax.grid(True, alpha=0.3, axis='x')
else:
    ax.text(0.5, 0.5, 'No correlation data', ha='center', va='center', 
            transform=ax.transAxes, fontsize=12)
    ax.axis('off')

# Plot 4: Train vs Test score (overfitting analysis)
ax = axes[3]
ax.scatter(cv_results['mean_train_score'], cv_results['mean_test_score'], 
          alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
min_score = min(cv_results['mean_train_score'].min(), cv_results['mean_test_score'].min())
max_score = max(cv_results['mean_train_score'].max(), cv_results['mean_test_score'].max())
ax.plot([min_score, max_score], [min_score, max_score], 'r--', linewidth=2, alpha=0.7, 
        label='Perfect Generalization')
ax.set_xlabel('Mean Train Score', fontsize=11)
ax.set_ylabel('Mean Test Score', fontsize=11)
ax.set_title('Overfitting Analysis', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plots 5-7: Effect of key hyperparameters (scatter plots)
key_params = ['max_depth', 'learning_rate', 'min_child_weight']
for plot_idx, param in enumerate(key_params):
    ax = axes[4 + plot_idx]
    param_col = f'param_{param}'
    if param_col in cv_results.columns:
        param_values = pd.to_numeric(cv_results[param_col], errors='coerce')
        valid_mask = ~param_values.isna()
        
        if valid_mask.sum() > 0:
            # Scatter plot with color gradient
            scatter = ax.scatter(param_values[valid_mask], 
                               cv_results.loc[valid_mask, 'mean_test_score'],
                               c=cv_results.loc[valid_mask, 'mean_test_score'],
                               cmap='viridis', s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
            
            # Highlight best value
            best_value = random_search.best_params_[param]
            best_mask = param_values == best_value
            if best_mask.sum() > 0:
                ax.scatter(param_values[best_mask], 
                          cv_results.loc[best_mask, 'mean_test_score'],
                          marker='*', s=400, color='red', edgecolors='black', 
                          linewidth=2, label='Best', zorder=5)
            
            ax.set_xlabel(param, fontsize=11)
            ax.set_ylabel('CV AUC-ROC', fontsize=11)
            ax.set_title(f'Effect of {param}', fontsize=11, fontweight='bold')
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            plt.colorbar(scatter, ax=ax, label='Score')
    else:
        ax.text(0.5, 0.5, f'No data for {param}', ha='center', va='center', 
               transform=ax.transAxes, fontsize=10)
        ax.axis('off')

# Plot 8: CV score stability (std vs mean)
ax = axes[7]
scatter = ax.scatter(cv_results['mean_test_score'], cv_results['std_test_score'],
                    alpha=0.6, s=50, edgecolors='black', linewidth=0.5, c='steelblue')
# Highlight best
best_idx = cv_results['rank_test_score'] == 1
ax.scatter(cv_results.loc[best_idx, 'mean_test_score'], 
          cv_results.loc[best_idx, 'std_test_score'],
          marker='*', s=400, color='red', edgecolors='black', linewidth=2, 
          label='Best', zorder=5)
ax.set_xlabel('Mean CV Score', fontsize=11)
ax.set_ylabel('Std CV Score', fontsize=11)
ax.set_title('Score Stability', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hyperparameter_tuning_analysis.png', dpi=300, bbox_inches='tight')
print(f"✓ Hyperparameter analysis saved to: hyperparameter_tuning_analysis.png")
plt.show()

# Additional visualization: Heatmap of parameter pairs (top 2 most important)
if len(param_correlations) >= 2:
    top_2_params = sorted(param_correlations.items(), key=lambda x: abs(x[1]), reverse=True)[:2]
    param1_name, param2_name = top_2_params[0][0], top_2_params[1][0]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    param1_col = f'param_{param1_name}'
    param2_col = f'param_{param2_name}'
    
    if param1_col in cv_results.columns and param2_col in cv_results.columns:
        param1_values = pd.to_numeric(cv_results[param1_col], errors='coerce')
        param2_values = pd.to_numeric(cv_results[param2_col], errors='coerce')
        scores = cv_results['mean_test_score']
        
        valid_mask = ~(param1_values.isna() | param2_values.isna())
        
        if valid_mask.sum() > 0:
            scatter = ax.scatter(param1_values[valid_mask], param2_values[valid_mask],
                               c=scores[valid_mask], s=100, cmap='RdYlGn', 
                               alpha=0.7, edgecolors='black', linewidth=1)
            
            # Highlight best
            best_param1 = random_search.best_params_[param1_name]
            best_param2 = random_search.best_params_[param2_name]
            ax.scatter([best_param1], [best_param2], marker='*', s=600, 
                      color='blue', edgecolors='black', linewidth=2, 
                      label='Best Combination', zorder=5)
            
            ax.set_xlabel(param1_name, fontsize=12)
            ax.set_ylabel(param2_name, fontsize=12)
            ax.set_title(f'Parameter Interaction: {param1_name} vs {param2_name}',
                        fontsize=13, fontweight='bold')
            ax.legend(fontsize=11)
            ax.grid(True, alpha=0.3)
            
            cbar = plt.colorbar(scatter, ax=ax, label='CV AUC-ROC')
            cbar.ax.tick_params(labelsize=10)
            
            plt.tight_layout()
            plt.savefig('hyperparameter_interaction.png', dpi=300, bbox_inches='tight')
            print(f"✓ Parameter interaction plot saved to: hyperparameter_interaction.png")
            plt.show()

# Extract best parameters for next step
best_params = random_search.best_params_
print("\n" + "="*80)
print("Best parameters will be used for final PU-Bagging model in Step 4")
print("="*80)


In [ ]:

# ==============================================================================
# STEP 4: CONFIGURE AND TRAIN PU-BAGGING CLASSIFIER
# ==============================================================================
print("\n" + "="*80)
print("STEP 4: PU-Bagging Classifier Training with Optimized Hyperparameters")
print("="*80)

# Configure base estimator with BEST parameters from Grid Search
base_estimator = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    min_child_weight=best_params['min_child_weight'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    gamma=0.1,                # Fixed: minimum loss reduction for split
    reg_alpha=best_params['reg_alpha'],
    reg_lambda=best_params['reg_lambda'],
    random_state=RANDOM_STATE,
    n_jobs=1,                 # Parallelism handled at bag level
    eval_metric='logloss',
    use_label_encoder=False
)

# base_estimator = xgb.XGBClassifier(
#     n_estimators=100,
#     max_depth=6,
#     learning_rate=0.05,
#     min_child_weight=1,
#     subsample=0.9,
#     colsample_bytree=0.9,
#     gamma=0.1,                # Fixed: minimum loss reduction for split
#     reg_alpha=0,
#     reg_lambda=0.5,
#     random_state=RANDOM_STATE,
#     n_jobs=1,                 # Parallelism handled at bag level
#     eval_metric='logloss',
#     use_label_encoder=False
# )
best_params=random_search.best_params_
print("Base estimator configuration (OPTIMIZED via Grid Search):")
print(f"  Type: XGBClassifier")
print(f"  Trees per bag: 100")
print(f"  Max depth: {best_params['max_depth']}")
print(f"  Learning rate: {best_params['learning_rate']}")
print(f"  Min child weight: {best_params['min_child_weight']}")
print(f"  Subsample: {best_params['subsample']}")
print(f"  Colsample by tree: {best_params['colsample_bytree']}")
print(f"  L1/L2 regularization: {best_params['reg_alpha']}/{best_params['reg_lambda']}")
# print(f"  Best CV AUC-ROC: {grid_search.best_score_:.4f}")

# Initialize PU-bagging classifier
pu_model = PUBaggingClassifier(
    base_estimator=base_estimator,
    n_bags=100,              # 100 bootstrap bags
    bag_size=0.7,            # Use 70% of unlabeled per bag
    positive_label=1,
    unlabeled_label=0,
    n_jobs=-1,               # Use all CPUs
    random_state=RANDOM_STATE,
    verbose=2
)

print("\nPU-Bagging configuration:")
print(f"  Number of bags: 100")
print(f"  Bag size: 70% of unlabeled samples")
print(f"  Strategy: All positives + bootstrapped unlabeled")
print(f"  Sample weighting: Using log-transformed tonnage weights")


# Train model WITH TONNAGE WEIGHTS

print("\nTraining PU-bagging model with tonnage-based sample weights...")
print(f"  Positive samples weight range: {weights_train[y_train==1].min():.3f} - {weights_train[y_train==1].max():.3f}")
print(f"  Unlabeled samples weight: 1.000 (uniform)")

pu_model.fit(X_train, y_train, sample_weight=weights_train)
print(f"\n✓ Model training complete with tonnage weighting")


In [ ]:
# ==============================================================================
# STEP 5: PREDICTION WITH UNCERTAINTY QUANTIFICATION
# ==============================================================================
print("\n" + "="*80)
print("STEP 5: Prediction with Uncertainty")
print("="*80)

# Predict with uncertainty
y_pred_mean, y_pred_std = pu_model.predict_with_uncertainty(X_test)

print(f"Prediction statistics:")
print(f"  Mean probability: {y_pred_mean.mean():.3f}")
print(f"  Std probability: {y_pred_mean.std():.3f}")
print(f"  Mean uncertainty (std across bags): {y_pred_std.mean():.3f}")
print(f"  Max uncertainty: {y_pred_std.max():.3f}")

# Compute performance metrics
auc_roc = roc_auc_score(y_test, y_pred_mean)
auc_pr = average_precision_score(y_test, y_pred_mean)

print(f"\n⚠ NOTE: These metrics treat unlabeled as negative (standard sklearn behavior)")
print(f"   In PU learning, unlabeled ≠ negative. See Step 5A for PU-aware metrics.")
print(f"\nModel performance on test set (reference only):")
print(f"  AUC-ROC: {auc_roc:.3f}")
print(f"  AUC-PR (Average Precision): {auc_pr:.3f}")

# Analyze predictions by class
print(f"\nPrediction distribution:")
print(f"  Positive samples - Mean: {y_pred_mean[y_test==1].mean():.3f}, Std: {y_pred_mean[y_test==1].std():.3f}")
print(f"  Unlabeled samples - Mean: {y_pred_mean[y_test==0].mean():.3f}, Std: {y_pred_mean[y_test==0].std():.3f}")

# Plot prediction distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of predictions by class
ax = axes[0]
ax.hist(y_pred_mean[y_test == 0], bins=50, alpha=0.6, label='Unlabeled', color='blue', density=True)
ax.hist(y_pred_mean[y_test == 1], bins=50, alpha=0.6, label='Positive (Known Deposits)', color='red', density=True)
ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Prediction Distribution by Class', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.text(0.02, 0.98, f'AUC-ROC: {auc_roc:.3f}\nAUC-PR: {auc_pr:.3f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Uncertainty vs prediction
ax = axes[1]
scatter = ax.scatter(y_pred_mean, y_pred_std, c=y_test, cmap='coolwarm', 
                    alpha=0.5, edgecolors='black', linewidth=0.3, s=20)
ax.set_xlabel('Mean Prediction', fontsize=11)
ax.set_ylabel('Prediction Uncertainty (Std)', fontsize=11)
ax.set_title('Prediction Uncertainty Analysis', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax, label='True Label')
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['Unlabeled', 'Positive'])

plt.tight_layout()
plt.savefig('pu_predictions_copper.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Prediction plots saved to: pu_predictions_copper.png")
plt.show()

In [ ]:
# ==============================================================================
# STEP 5A: MINERAL PROSPECTIVITY EVALUATION METRICS (PU-AWARE)
# ==============================================================================
print("\n" + "="*80)
print("STEP 5A: Mineral Prospectivity Evaluation Metrics")
print("="*80)

print("\n⚠ IMPORTANT: PU Learning Evaluation Strategy")
print("  In PU (Positive-Unlabeled) learning:")
print("  - Positive samples (y=1) = KNOWN deposits")
print("  - Unlabeled samples (y=0) = UNKNOWN (may contain hidden deposits)")
print("  - We evaluate ONLY on known deposits' rankings")
print("  - We do NOT penalize high predictions for unlabeled samples")

# Get predictions for ALL test samples and known deposits separately
n_total = len(y_pred_mean)
n_deposits = np.sum(y_test == 1)
deposit_mask = y_test == 1

print(f"\nDataset statistics:")
print(f"  Total test samples: {n_total:,}")
print(f"  Known deposits (positives): {n_deposits}")
print(f"  Unlabeled samples: {n_total - n_deposits:,}")
print(f"  Known deposit rate: {n_deposits/n_total*100:.2f}%")

# Get predictions for deposits only
deposit_predictions = y_pred_mean[deposit_mask]
deposit_indices = np.where(deposit_mask)[0]

# Rank ALL samples by prediction score (descending)
all_ranked_indices = np.argsort(y_pred_mean)[::-1]
all_ranked_predictions = y_pred_mean[all_ranked_indices]

# For each known deposit, find its rank among ALL predictions
deposit_ranks = []
for dep_idx in deposit_indices:
    rank = np.where(all_ranked_indices == dep_idx)[0][0] + 1  # 1-indexed rank
    deposit_ranks.append({
        'deposit_idx': dep_idx,
        'rank': rank,
        'prediction': y_pred_mean[dep_idx],
        'percentile': (rank / n_total) * 100
    })

deposit_ranks_df = pd.DataFrame(deposit_ranks)

print(f"\nDeposit ranking statistics:")
print(f"  Best ranked deposit: Rank {deposit_ranks_df['rank'].min()} ({deposit_ranks_df['percentile'].min():.2f}% percentile)")
print(f"  Median ranked deposit: Rank {deposit_ranks_df['rank'].median():.0f} ({deposit_ranks_df['percentile'].median():.2f}% percentile)")
print(f"  Worst ranked deposit: Rank {deposit_ranks_df['rank'].max():.0f} ({deposit_ranks_df['percentile'].max():.2f}% percentile)")
print(f"  Mean deposit prediction: {deposit_predictions.mean():.3f}")
print(f"  Median deposit prediction: {np.median(deposit_predictions):.3f}")

# ==============================================================================
# 1. RECALL@K - HOW MANY DEPOSITS IN TOP-K% OF RANKED AREA
# ==============================================================================
print("\n" + "-"*80)
print("Recall@K: Deposits Captured in Top-K% Ranked Areas")
print("-"*80)

# Define key percentiles for mineral prospectivity
percentiles = [1, 2, 5, 10, 20]
metrics_at_k = []

for pct in percentiles:
    k = int(n_total * pct / 100)
    if k == 0:
        k = 1
    
    # Get top-k ranked indices (from ALL samples)
    top_k_indices = all_ranked_indices[:k]
    
    # Count how many of these are known deposits
    deposits_in_top_k = np.sum(deposit_mask[top_k_indices])
    
    # Recall@K: proportion of all known deposits captured in top-k
    recall_k = deposits_in_top_k / n_deposits if n_deposits > 0 else 0
    
    # Enrichment factor: how many times more deposits than random
    # If we sample K% of area randomly, we'd expect K% of deposits
    expected_random = n_deposits * (pct / 100)
    enrichment = deposits_in_top_k / expected_random if expected_random > 0 else 0
    
    metrics_at_k.append({
        'Area %': pct,
        'K (samples)': k,
        'Deposits Captured': int(deposits_in_top_k),
        'Recall@K': recall_k,
        'Enrichment': enrichment
    })
    
    print(f"\nTop {pct}% of ranked area (K={k:,}):")
    print(f"  Deposits captured: {int(deposits_in_top_k)} / {n_deposits} ({recall_k*100:.1f}%)")
    print(f"  Recall@K: {recall_k:.4f}")
    print(f"  Enrichment over random: {enrichment:.2f}x")

# Create DataFrame for metrics
metrics_df = pd.DataFrame(metrics_at_k)
print("\n" + "="*80)
print("Summary Table: Recall@K and Enrichment (PU-Aware Metrics)")
print("="*80)
print(metrics_df.to_string(index=False))

# Highlight key finding (typically top 5%)
top5_metrics = metrics_df[metrics_df['Area %'] == 5].iloc[0]
print("\n" + "="*80)
print("KEY FINDING (PU-AWARE EVALUATION)")
print("="*80)
print(f"The top 5% of ranked area captures {int(top5_metrics['Deposits Captured'])} out of {n_deposits} "
      f"known mineral occurrences ({top5_metrics['Recall@K']*100:.1f}%), "
      f"corresponding to an enrichment of {top5_metrics['Enrichment']:.1f}x over random expectation.")
print("\nNote: This evaluation focuses on ranking known deposits highly.")
print("      High-scoring unlabeled samples are NOT penalized (they may be undiscovered deposits).")
print("="*80)

# ==============================================================================
# 2. SUCCESS-RATE CURVES (CUMULATIVE KNOWN DEPOSITS VS AREA)
# ==============================================================================
print("\n" + "-"*80)
print("Computing Success-Rate Curves (Based on Known Deposits)")
print("-"*80)

# Compute cumulative known deposits captured as we go down the ranked list
y_test_array = y_test.values if hasattr(y_test, 'values') else y_test
cumulative_deposits = np.cumsum(y_test_array[all_ranked_indices])
cumulative_area_pct = np.arange(1, n_total + 1) / n_total * 100

# Random baseline (expected deposits if sampling randomly)
random_baseline = np.linspace(0, n_deposits, n_total)

# Compute success rate at various area percentages
success_rate_points = []
for area_pct in [1, 2, 5, 10, 20, 30, 50, 75, 100]:
    idx = int(n_total * area_pct / 100) - 1
    if idx >= len(cumulative_deposits):
        idx = len(cumulative_deposits) - 1
    
    deposits_at_pct = cumulative_deposits[idx]
    success_rate = (deposits_at_pct / n_deposits * 100) if n_deposits > 0 else 0
    
    success_rate_points.append({
        'Area %': area_pct,
        'Deposits Found': int(deposits_at_pct),
        'Success Rate %': success_rate
    })

success_rate_df = pd.DataFrame(success_rate_points)
print("\nSuccess Rate at Key Area Percentiles:")
print(success_rate_df.to_string(index=False))

# ==============================================================================
# 3. VISUALIZATION: SUCCESS-RATE CURVE AND LIFT CURVE
# ==============================================================================
print("\n" + "-"*80)
print("Generating Mineral Prospectivity Evaluation Plots")
print("-"*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Success-Rate Curve (Cumulative Deposits vs Area)
ax = axes[0, 0]
ax.plot(cumulative_area_pct, cumulative_deposits, 'b-', linewidth=2.5, label='PU Model', zorder=3)
ax.plot(cumulative_area_pct, random_baseline, 'r--', linewidth=1.5, label='Random Baseline', alpha=0.7)
ax.fill_between(cumulative_area_pct, random_baseline, cumulative_deposits, 
                 where=(cumulative_deposits >= random_baseline), 
                 alpha=0.3, color='green', label='Model Gain')
ax.set_xlabel('Cumulative Area Explored (%)', fontsize=12)
ax.set_ylabel('Cumulative Deposits Found', fontsize=12)
ax.set_title('Success-Rate Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
# ax.grid(True, alpha=0.3)
ax.set_xlim([0, 100])
ax.set_ylim([0, n_deposits * 1.05])

# Add annotation for top 5%
top5_idx = int(n_total * 0.05)
top5_deposits = cumulative_deposits[top5_idx - 1]
ax.plot([5, 5], [0, top5_deposits], 'g--', linewidth=1.5, alpha=0.7)
ax.plot([0, 5], [top5_deposits, top5_deposits], 'g--', linewidth=1.5, alpha=0.7)
ax.text(5, top5_deposits + n_deposits*0.02, 
        f'{int(top5_deposits)} deposits\nat 5% area', 
        fontsize=9, color='green', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='green', alpha=0.8))

# Plot 2: Lift Curve
ax = axes[0, 1]
lift_curve = (cumulative_deposits / np.arange(1, n_total + 1)) / (n_deposits / n_total)
# Only plot up to 50% area for clarity (lift becomes less meaningful at high coverage)
plot_limit = int(n_total * 0.5)
ax.plot(cumulative_area_pct[:plot_limit], lift_curve[:plot_limit], 'purple', linewidth=2.5)
ax.axhline(y=1, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Random (Lift=1)')
ax.set_xlabel('Cumulative Area Explored (%)', fontsize=12)
ax.set_ylabel('Lift over Random', fontsize=12)
ax.set_title('Lift Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
# ax.grid(True, alpha=0.3)
ax.set_xlim([0, 50])

# Add annotations for key percentiles
for pct in [1, 2, 5, 10]:
    idx = int(n_total * pct / 100) - 1
    lift_val = lift_curve[idx]
    ax.plot(pct, lift_val, 'ro', markersize=8, zorder=5)
    ax.text(pct, lift_val + 0.5, f'{lift_val:.1f}x\n@{pct}%', 
            fontsize=8, ha='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.6))

# Plot 3: Recall@K and Enrichment Curves (PU-Aware)
ax = axes[1, 0]
area_range = np.linspace(0, 30, 100)
recall_curve = []
enrichment_curve = []

for pct in area_range:
    k = int(n_total * pct / 100)
    if k == 0 or pct == 0:
        recall_curve.append(0)
        enrichment_curve.append(0)
    else:
        top_k_indices = all_ranked_indices[:k]
        deposits_in_top_k = np.sum(y_test_array[top_k_indices])
        recall_curve.append(deposits_in_top_k / n_deposits if n_deposits > 0 else 0)
        expected_random = n_deposits * (pct / 100)
        enrichment_curve.append(deposits_in_top_k / expected_random if expected_random > 0 else 0)

ax.plot(area_range, recall_curve, 'r-', linewidth=2.5, label='Recall@K (Known Deposits)', marker='s', 
        markevery=10, markersize=5)
ax2 = ax.twinx()
ax2.plot(area_range, enrichment_curve, 'purple', linewidth=2.5, label='Enrichment Factor', marker='o', 
         markevery=10, markersize=5, alpha=0.7)
ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, label='Random (Enrichment=1)')
ax.set_xlabel('Top Area (%) Selected', fontsize=12)
ax.set_ylabel('Recall@K', fontsize=12, color='red')
ax2.set_ylabel('Enrichment Factor', fontsize=12, color='purple')
ax.set_title('Recall@K and Enrichment', fontsize=13, fontweight='bold')
ax.tick_params(axis='y', labelcolor='red')
ax2.tick_params(axis='y', labelcolor='purple')
ax.legend(fontsize=10, loc='upper left')
ax2.legend(fontsize=10, loc='upper right')
# ax.grid(True, alpha=0.3)
ax.set_xlim([0, 30])
ax.set_ylim([0, 1.05])

# Add vertical lines for key percentiles
for pct in [1, 2, 5, 10]:
    ax.axvline(x=pct, color='green', linestyle=':', alpha=0.5, linewidth=1)
    ax.text(pct, 0.02, f'{pct}%', fontsize=8, ha='center', rotation=90,
            color='green', fontweight='bold')

# Plot 4: Summary Bar Chart - Deposits Captured at Key Percentiles
ax = axes[1, 1]
bar_data = metrics_df[metrics_df['Area %'] <= 20]  # Focus on top 20%
x_pos = np.arange(len(bar_data))
bars = ax.bar(x_pos, bar_data['Deposits Captured'], alpha=0.8, 
              edgecolor='black', linewidth=1.5, color='steelblue')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{int(pct)}%" for pct in bar_data['Area %']], fontsize=11)
ax.set_xlabel('Top Area Percentile', fontsize=12)
ax.set_ylabel('Number of Deposits Captured', fontsize=12)
ax.set_title('Deposits Captured at Key Percentiles\n(Out of {} Total)'.format(n_deposits), 
             fontsize=13, fontweight='bold')
# ax.grid(True, alpha=0.3, axis='y')

# Add value labels and recall percentage on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    recall_val = bar_data.iloc[i]['Recall@K']
    enrichment_val = bar_data.iloc[i]['Enrichment']
    
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.3,
            f'{int(height)}\n({recall_val*100:.1f}%)',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Add enrichment annotation inside bar
    ax.text(bar.get_x() + bar.get_width()/2., height/2,
            f'Enrich: {enrichment_val:.1f}x',
            ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.set_ylim([0, n_deposits * 1.15])

plt.tight_layout()
plt.savefig('mineral_prospectivity_metrics_copper_PU_aware.png', dpi=300, bbox_inches='tight')
print(f"\n✓ PU-aware mineral prospectivity evaluation plots saved to: mineral_prospectivity_metrics_copper_PU_aware.png")
plt.show()

# Save metrics to CSV
metrics_df.to_csv('prospectivity_metrics_summary_PU_aware.csv', index=False)
success_rate_df.to_csv('success_rate_curve_data_PU_aware.csv', index=False)
deposit_ranks_df.to_csv('deposit_rankings_PU_aware.csv', index=False)
print(f"✓ PU-aware metrics saved to: prospectivity_metrics_summary_PU_aware.csv")
print(f"✓ Success-rate data saved to: success_rate_curve_data_PU_aware.csv")
print(f"✓ Deposit rankings saved to: deposit_rankings_PU_aware.csv")

print("\n" + "="*80)
print("Mineral Prospectivity Evaluation Complete (PU-Aware)")
print("="*80)
print("\nKey Points:")
print("  ✓ Evaluation focuses on ranking known deposits highly")
print("  ✓ Unlabeled samples are NOT treated as negatives")
print("  ✓ Enrichment factor replaces 'lift' for PU learning context")
print("  ✓ High-scoring unlabeled areas may contain undiscovered deposits")

In [ ]:
# ==============================================================================
# STEP 5B: IDENTIFY POTENTIAL UNDISCOVERED DEPOSITS (HIGH-SCORING UNLABELED)
# ==============================================================================
print("\n" + "="*80)
print("STEP 5B: High-Prospectivity Unlabeled Areas (Potential Discoveries)")
print("="*80)

print("\nIn PU learning, high-scoring unlabeled samples may represent:")
print("  1. Undiscovered mineral deposits")
print("  2. Areas with similar geological characteristics to known deposits")
print("  3. Priority targets for future exploration")

# Get unlabeled samples and their predictions
unlabeled_mask = y_test == 0
n_unlabeled = np.sum(unlabeled_mask)
unlabeled_predictions = y_pred_mean[unlabeled_mask]

# Define high-prospectivity thresholds based on deposit prediction distribution
deposit_prediction_threshold_90 = np.percentile(deposit_predictions, 10)  # Lower 10% of deposits
deposit_prediction_threshold_50 = np.percentile(deposit_predictions, 50)  # Median deposit score
deposit_prediction_threshold_25 = np.percentile(deposit_predictions, 75)  # Upper 25% of deposits

print(f"\nKnown deposit prediction score benchmarks:")
print(f"  10th percentile (lower deposits): {deposit_prediction_threshold_90:.3f}")
print(f"  50th percentile (median deposit): {deposit_prediction_threshold_50:.3f}")
print(f"  75th percentile (high-quality deposits): {deposit_prediction_threshold_25:.3f}")

# Count unlabeled samples exceeding these thresholds
unlabeled_above_90 = np.sum(unlabeled_predictions >= deposit_prediction_threshold_90)
unlabeled_above_50 = np.sum(unlabeled_predictions >= deposit_prediction_threshold_50)
unlabeled_above_25 = np.sum(unlabeled_predictions >= deposit_prediction_threshold_25)

print(f"\nUnlabeled samples with deposit-like characteristics:")
print(f"  Above 10th percentile of deposits: {unlabeled_above_90:,} ({unlabeled_above_90/n_unlabeled*100:.2f}%)")
print(f"  Above median deposit score: {unlabeled_above_50:,} ({unlabeled_above_50/n_unlabeled*100:.2f}%)")
print(f"  Above 75th percentile of deposits: {unlabeled_above_25:,} ({unlabeled_above_25/n_unlabeled*100:.2f}%)")

# Analyze top-ranked unlabeled areas
top_k_pct = 5  # Top 5% of ALL ranked areas
top_k = int(n_total * top_k_pct / 100)
top_k_indices = all_ranked_indices[:top_k]

# Count deposits vs unlabeled in top-K
top_k_deposits = np.sum(deposit_mask[top_k_indices])
top_k_unlabeled = np.sum(unlabeled_mask[top_k_indices])

print(f"\nComposition of top {top_k_pct}% ranked areas (K={top_k:,} samples):")
print(f"  Known deposits: {top_k_deposits} ({top_k_deposits/top_k*100:.1f}%)")
print(f"  Unlabeled (potential targets): {top_k_unlabeled} ({top_k_unlabeled/top_k*100:.1f}%)")

# Get indices of high-prospectivity unlabeled samples in test set
high_prosp_threshold = deposit_prediction_threshold_50  # Use median deposit score
high_prosp_unlabeled_mask = unlabeled_mask & (y_pred_mean >= high_prosp_threshold)
n_high_prosp_unlabeled = np.sum(high_prosp_unlabeled_mask)

print(f"\n" + "="*80)
print(f"EXPLORATION RECOMMENDATION")
print("="*80)
print(f"Identified {n_high_prosp_unlabeled:,} high-prospectivity unlabeled locations")
print(f"(prediction score ≥ {high_prosp_threshold:.3f}, matching median known deposit)")
print(f"\nThese areas should be prioritized for:")
print(f"  • Detailed geological surveys")
print(f"  • Geophysical/geochemical sampling")
print(f"  • Exploratory drilling")
print("="*80)

# Visualization: Compare prediction distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Histogram comparison
ax = axes[0]
bins = np.linspace(0, 1, 50)
ax.hist(deposit_predictions, bins=bins, alpha=0.7, label=f'Known Deposits (n={n_deposits})', 
        color='red', density=True, edgecolor='black', linewidth=0.5)
ax.hist(unlabeled_predictions, bins=bins, alpha=0.5, label=f'Unlabeled (n={n_unlabeled:,})', 
        color='blue', density=True, edgecolor='black', linewidth=0.5)

# Add threshold lines
ax.axvline(deposit_prediction_threshold_50, color='darkred', linestyle='--', linewidth=2, 
           label=f'Median Deposit Score ({deposit_prediction_threshold_50:.3f})')
ax.axvline(deposit_prediction_threshold_25, color='purple', linestyle='--', linewidth=2, 
           label=f'75th %ile Deposit Score ({deposit_prediction_threshold_25:.3f})')

ax.set_xlabel('Prediction Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Prediction Distribution: Known Deposits vs Unlabeled', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

# Highlight overlap region
overlap_start = deposit_prediction_threshold_90
ax.axvspan(overlap_start, 1.0, alpha=0.2, color='yellow', 
           label=f'{unlabeled_above_90:,} unlabeled in deposit range')
ax.text(0.7, ax.get_ylim()[1]*0.9, 
        f'{unlabeled_above_50:,} unlabeled\nabove median\ndeposit score', 
        fontsize=10, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# Plot 2: Cumulative distribution
ax = axes[1]
sorted_deposit_preds = np.sort(deposit_predictions)
sorted_unlabeled_preds = np.sort(unlabeled_predictions)

ax.plot(sorted_deposit_preds, np.linspace(0, 1, len(sorted_deposit_preds)), 
        'r-', linewidth=2.5, label='Known Deposits (CDF)')
ax.plot(sorted_unlabeled_preds, np.linspace(0, 1, len(sorted_unlabeled_preds)), 
        'b-', linewidth=2.5, alpha=0.7, label='Unlabeled (CDF)')

ax.axvline(deposit_prediction_threshold_50, color='darkred', linestyle='--', linewidth=2, alpha=0.7)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)

ax.set_xlabel('Prediction Score', fontsize=12)
ax.set_ylabel('Cumulative Probability', fontsize=12)
ax.set_title('Cumulative Distribution: Identifying High-Prospectivity Unlabeled', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate the separation
ax.text(deposit_prediction_threshold_50 + 0.05, 0.25, 
        f'{unlabeled_above_50:,} unlabeled samples\nexceed median deposit score\n(potential discoveries)', 
        fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('potential_undiscovered_deposits.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Potential undiscovered deposits analysis saved to: potential_undiscovered_deposits.png")
plt.show()

# Save high-prospectivity unlabeled locations for follow-up
if hasattr(X_test, 'index'):
    high_prosp_indices = X_test.index[high_prosp_unlabeled_mask]
else:
    high_prosp_indices = np.where(high_prosp_unlabeled_mask)[0]

high_prosp_df = pd.DataFrame({
    'test_index': high_prosp_indices,
    'prediction_score': y_pred_mean[high_prosp_unlabeled_mask],
    'rank_percentile': [(np.where(all_ranked_indices == idx)[0][0] + 1) / n_total * 100 
                        for idx in np.where(high_prosp_unlabeled_mask)[0]]
})
high_prosp_df = high_prosp_df.sort_values('prediction_score', ascending=False)
high_prosp_df.to_csv('high_prospectivity_unlabeled_targets.csv', index=False)
print(f"✓ High-prospectivity unlabeled targets saved to: high_prospectivity_unlabeled_targets.csv")
print(f"\n  Top 10 highest-scoring unlabeled targets:")
print(high_prosp_df.head(10).to_string(index=False))

print("\n" + "="*80)

In [ ]:
# ==============================================================================
# STEP 6: FEATURE IMPORTANCE ANALYSIS
# ==============================================================================
print("\n" + "="*80)
print("STEP 6: Feature Importance Analysis")
print("="*80)

# Get feature importances across bags
importance_mean, importance_std = pu_model.get_feature_importances()

# Create importance DataFrame
importance_df = pd.DataFrame({
    'feature': X_clean.columns,
    'importance_mean': importance_mean,
    'importance_std': importance_std,
    'cv': importance_std / (importance_mean + 1e-10)  # Coefficient of variation
})

# Assign geological groups using the same patterns from Step 1.5
# First-match logic: more specific patterns checked first
group_mapping = {}
for feature in X_clean.columns:
    assigned = False
    # Check each group in order (order matters!)
    for group_name, patterns in feature_groups_patterns.items():
        for pattern in patterns:
            # Convert glob pattern to regex
            regex_pattern = pattern.replace('*', '.*')
            if re.match(f'^{regex_pattern}$', feature, re.IGNORECASE):
                group_mapping[feature] = group_name
                assigned = True
                break
        if assigned:
            break
    
    if not assigned:
        group_mapping[feature] = 'ungrouped'

importance_df['group'] = importance_df['feature'].map(group_mapping)

# Sort by importance
importance_df = importance_df.sort_values('importance_mean', ascending=False)

print(f"\nTop 20 most important features:")
print(importance_df.head(5)[['feature', 'importance_mean', 'importance_std', 'group']].to_string(index=False))

# Save full report
importance_df.to_csv('feature_importance_report.csv', index=False)
print(f"\n✓ Full importance report saved to: feature_importance_report.csv")

# Plot top features
fig, axes = plt.subplots(1, 2, figsize=(14, 6))


def clean_feature_name(feature):
    name = feature

    # Remove long prefixes
    name = name.replace("Geophysics", "")
    name = name.replace("Litho_", "Lithology: ")
    
    # Gravity
    name = name.replace("Gravity", "Bouguer Gravity Anomaly")
    name = name.replace("Up30km", "\n(upward continued 30 km)")
    
    # Magnetics
    name = name.replace("Mag", "Magnetic Anomaly")
    name = name.replace("Mag_RTP_VD", "Magnetic RTP vertical \nderivative")
    name = name.replace("Mag_RTP", "Magnetic RTP")

    
    # Gradients
    name = name.replace("_x_grad_", "  \nX-gradient ")
    name = name.replace("_y_grad_", "  \nY-gradient ")
    name = name.replace("_grad_", " \ngradient ")
    
    # Statistics
    name = name.replace("_std", " (std)")
    name = name.replace("_max", " (maximum)")
    name = name.replace("_min", " (minimum)")
    
    # Distance
    name = name.replace("nearest_distance", " \ndistance to nearest unit")
    
    # Clean underscores
    name = name.replace("_", " ")
    
    # Capitalize first letter
    name = name.strip()
    name = name[0].upper() + name[1:]
    
    return name


# for column in importance_df.columns:
#     new_col=format_label(column)
#     importance_df=importance_df.rename(columns={column:new_col})
# Bar plot of top features
ax = axes[0]
top_n = 10
top_features = importance_df.head(top_n)


y_pos = np.arange(len(top_features))
colors = plt.cm.viridis(np.linspace(0, 1, top_n))
ax.barh(y_pos, top_features['importance_mean'], 
        xerr=top_features['importance_std'],
        alpha=0.8, edgecolor='black', color=colors, linewidth=0.5)
ax.set_yticks(y_pos)
# ax.set_yticklabels(top_features['feature'], fontsize=7)
top_features['clean_name'] = top_features['feature'].apply(clean_feature_name)

ax.set_yticklabels(top_features['clean_name'], fontsize=8)
ax.set_xlabel('Mean Importance', fontsize=11)
ax.set_title(f'Top {top_n} Most Important Features', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Group-level importance
ax = axes[1]
group_importance = importance_df.groupby('group')['importance_mean'].sum().sort_values(ascending=False)
colors_group = plt.cm.Set3(np.linspace(0, 1, len(group_importance)))
bars = ax.bar(range(len(group_importance)), group_importance.values, 
       alpha=0.8, edgecolor='black', color=colors_group, linewidth=0.8)
ax.set_xticks(range(len(group_importance)))
ax.set_xticklabels(group_importance.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Total Importance', fontsize=11)
ax.set_title('Feature Importance by Geological Group', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, group_importance.values)):
    ax.text(i, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('feature_importance_copper.png', dpi=600, bbox_inches='tight')
print(f"\n✓ Feature importance plots saved to: feature_importance_copper.png")
print(f"\nGroup-level importance summary:")
for group, importance in group_importance.items():
    n_features = (importance_df['group'] == group).sum()
    print(f"  {group}: {importance:.3f} ({n_features} features)")

print(f"  {group}: {importance:.3f} ({n_features} features)")
for group, importance in group_importance.items():    n_features = (importance_df['group'] == group).sum()


In [ ]:
# ==============================================================================
# STEP 7: POST-PU DIAGNOSTICS WITH SHAP
# ==============================================================================
print("\n" + "="*80)
print("STEP 7: Post-PU Feature Diagnostics")
print("="*80)

# Initialize diagnostics
diagnostics = PUFeatureDiagnostics(
    pu_model=pu_model,
    feature_names=X_clean.columns.tolist(),
    feature_groups=feature_groups_patterns,
    verbose=True
)

# Compute feature selection frequency
selection_freq = diagnostics.compute_feature_selection_frequency()

# Compute SHAP values (subsample for speed with large dataset)
print("\nComputing SHAP values for interpretability...")
print("Note: This may take 5-10 minutes for large datasets...")

try:
    # Use 500 samples for SHAP computation (balance between speed and accuracy)
    n_shap_samples = min(500, len(X_test))
    
    shap_values = diagnostics.compute_shap_values(
        X_test, 
        n_samples=n_shap_samples,
        random_state=RANDOM_STATE
    )
    
    # Compute SHAP sign consistency
    sign_consistency = diagnostics.compute_shap_sign_consistency()
    
    # Get comprehensive stability report
    stability_report = diagnostics.get_feature_stability_report()
    
    print(f"\nFeature stability report:")
    print(f"  Total features: {len(stability_report)}")
    print(f"  Highly stable (score > 0.8): {(stability_report['stability_score'] > 0.8).sum()}")
    print(f"  Moderately stable (0.5-0.8): {((stability_report['stability_score'] >= 0.5) & (stability_report['stability_score'] <= 0.8)).sum()}")
    print(f"  Unstable (score < 0.5): {(stability_report['stability_score'] < 0.5).sum()}")
    
    print(f"\nTop 15 most stable features:")
    print(stability_report.head(15)[['feature', 'selection_freq', 'shap_sign_consistency', 'stability_score', 'group']].to_string(index=False))
    
    # Identify unstable features
    unstable_features = diagnostics.identify_unstable_features(
        shap_sign_threshold=0.7,
        selection_freq_threshold=0.3
    )
    
    if len(unstable_features) > 0:
        print(f"\n⚠ Warning: {len(unstable_features)} unstable features identified")
        print(f"Consider removing these features in future iterations for improved stability")
        print(f"\nMost unstable features:")
        unstable_report = stability_report[stability_report['feature'].isin(unstable_features)].head(10)
        print(unstable_report[['feature', 'selection_freq', 'shap_sign_consistency', 'flag']].to_string(index=False))
    
    # Plot diagnostics
    print("\nGenerating stability diagnostic plots...")
    diagnostics.plot_feature_stability(
        top_n=25,
        save_path='feature_stability_copper.png'
    )
    print(f"✓ Stability diagnostics saved to: feature_stability_copper.png")
    
    # Save stability report
    stability_report.to_csv('feature_stability_report_copper.csv', index=False)
    print(f"✓ Stability report saved to: feature_stability_report_copper.csv")
    
    shap_success = True

except ImportError as e:
    print(f"\n⚠ SHAP not available: {e}")
    print("Install SHAP for full interpretability: pip install shap")
    print("Continuing with basic feature selection frequency only...")
    shap_success = False
    
except Exception as e:
    print(f"\n⚠ Error computing SHAP values: {e}")
    print("Continuing without SHAP diagnostics...")
    shap_success = False

In [ ]:
# ==============================================================================
# STEP 7.5: MODEL PERFORMANCE VS DEPOSIT TONNAGE ANALYSIS
# ==============================================================================
print("\n" + "="*80)
print("STEP 7.5: Analyze Model Performance by Deposit Tonnage")
print("="*80)

# Get tonnage values for test set deposits
test_deposit_mask = y_test == 1
test_indices_deposits = X_test.index[test_deposit_mask]
test_deposit_tonnages = ALL_DATA.loc[test_indices_deposits, 'tonnage_mt'].values
test_deposit_predictions = y_pred_mean[test_deposit_mask]

# Filter out deposits without tonnage data
valid_tonnage_mask = ~np.isnan(test_deposit_tonnages)
if valid_tonnage_mask.sum() == 0:
    print("⚠ No tonnage data available for test deposits")
else:
    valid_tonnages = test_deposit_tonnages[valid_tonnage_mask]
    valid_predictions = test_deposit_predictions[valid_tonnage_mask]
    
    print(f"\nTest deposits with tonnage data: {len(valid_tonnages)}")
    print(f"Tonnage range: {valid_tonnages.min():.2f} - {valid_tonnages.max():.2f} Mt")
    
    # Create tonnage bins (quartiles)
    quartiles = np.percentile(valid_tonnages, [25, 50, 75])
    tonnage_bins = ['Very Low (<Q1)', 'Low (Q1-Q2)', 'Medium (Q2-Q3)', 'High (>Q3)']
    
    bin_assignments = np.digitize(valid_tonnages, quartiles)
    
    print(f"\nTonnage quartiles:")
    print(f"  Q1 (25th): {quartiles[0]:.2f} Mt")
    print(f"  Q2 (50th/Median): {quartiles[1]:.2f} Mt")
    print(f"  Q3 (75th): {quartiles[2]:.2f} Mt")
    
    # Calculate performance metrics by tonnage bin
    print(f"\n{'='*80}")
    print("MODEL PERFORMANCE BY DEPOSIT SIZE")
    print(f"{'='*80}")
    
    bin_stats = []
    for i, bin_name in enumerate(tonnage_bins):
        mask = bin_assignments == i
        if mask.sum() > 0:
            bin_tonnages = valid_tonnages[mask]
            bin_preds = valid_predictions[mask]
            
            stats = {
                'Bin': bin_name,
                'Count': mask.sum(),
                'Tonnage_Min': bin_tonnages.min(),
                'Tonnage_Max': bin_tonnages.max(),
                'Tonnage_Mean': bin_tonnages.mean(),
                'Pred_Mean': bin_preds.mean(),
                'Pred_Std': bin_preds.std(),
                'Pred_Median': np.median(bin_preds),
                'High_Conf_Rate': (bin_preds > 0.7).sum() / len(bin_preds)
            }
            bin_stats.append(stats)
            
            print(f"\n{bin_name}:")
            print(f"  Deposits: {stats['Count']}")
            print(f"  Tonnage range: {stats['Tonnage_Min']:.2f} - {stats['Tonnage_Max']:.2f} Mt")
            print(f"  Mean tonnage: {stats['Tonnage_Mean']:.2f} Mt")
            print(f"  Mean prediction: {stats['Pred_Mean']:.3f}")
            print(f"  Median prediction: {stats['Pred_Median']:.3f}")
            print(f"  Prediction std: {stats['Pred_Std']:.3f}")
            print(f"  High confidence rate (>0.7): {stats['High_Conf_Rate']:.1%}")
    
    bin_stats_df = pd.DataFrame(bin_stats)
    
    # Visualize performance vs tonnage
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Scatter: Tonnage vs Prediction
    ax = axes[0, 0]
    scatter = ax.scatter(valid_tonnages, valid_predictions, 
                        c=valid_predictions, cmap='RdYlGn', 
                        s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
    ax.set_xlabel('Deposit Tonnage (Mt)', fontsize=11)
    ax.set_ylabel('Model Prediction Score', fontsize=11)
    ax.set_title('Model Performance vs Deposit Size', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Threshold=0.5')
    ax.axhline(y=0.7, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='High Conf=0.7')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    plt.colorbar(scatter, ax=ax, label='Prediction Score')
    
    # Add correlation coefficient
    from scipy.stats import spearmanr
    corr, pval = spearmanr(valid_tonnages, valid_predictions)
    ax.text(0.02, 0.98, f'Spearman ρ = {corr:.3f}\np-value = {pval:.3e}',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 2. Box plot by tonnage bin
    ax = axes[0, 1]
    bin_data = [valid_predictions[bin_assignments == i] for i in range(len(tonnage_bins))]
    bp = ax.boxplot(bin_data, labels=tonnage_bins, patch_artist=True,
                    medianprops=dict(color='red', linewidth=2))
    
    # Color boxes by performance
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(tonnage_bins)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_ylabel('Prediction Score', fontsize=11)
    ax.set_title('Prediction Distribution by Tonnage Quartile', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
    
    # 3. Bar chart: Mean prediction by bin
    ax = axes[1, 0]
    bars = ax.bar(range(len(bin_stats_df)), bin_stats_df['Pred_Mean'], 
                  yerr=bin_stats_df['Pred_Std'], capsize=8,
                  color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    ax.set_xticks(range(len(bin_stats_df)))
    ax.set_xticklabels(bin_stats_df['Bin'], rotation=15, ha='right')
    ax.set_ylabel('Mean Prediction Score', fontsize=11)
    ax.set_title('Mean Model Performance by Deposit Size', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)
    
    # Add value labels
    for i, (bar, val, count) in enumerate(zip(bars, bin_stats_df['Pred_Mean'], bin_stats_df['Count'])):
        ax.text(i, val + 0.02, f'{val:.3f}\n(n={count})', 
               ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # 4. High confidence rate by bin
    ax = axes[1, 1]
    bars = ax.bar(range(len(bin_stats_df)), bin_stats_df['High_Conf_Rate'], 
                  color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    ax.set_xticks(range(len(bin_stats_df)))
    ax.set_xticklabels(bin_stats_df['Bin'], rotation=15, ha='right')
    ax.set_ylabel('High Confidence Rate (>0.7)', fontsize=11)
    ax.set_title('High Confidence Detection by Deposit Size', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    for i, (bar, val, count) in enumerate(zip(bars, bin_stats_df['High_Conf_Rate'], bin_stats_df['Count'])):
        ax.text(i, val + 0.02, f'{val:.1%}', 
               ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('performance_vs_tonnage_analysis.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Performance vs tonnage plots saved to: performance_vs_tonnage_analysis.png")
    plt.show()
    
    # Save statistics
    bin_stats_df.to_csv('performance_by_tonnage_bins.csv', index=False)
    print(f"✓ Tonnage bin statistics saved to: performance_by_tonnage_bins.csv")
    
    # Statistical test for trend
    print(f"\n{'='*80}")
    print("INTERPRETATION:")
    print(f"{'='*80}")
    if corr > 0.2 and pval < 0.05:
        print("✓ POSITIVE CORRELATION: Model performs BETTER on larger deposits")
        print("  → Large deposits are easier to detect (stronger signals)")
        print("  → Tonnage weighting may be reinforcing this bias")
    elif corr < -0.2 and pval < 0.05:
        print("✓ NEGATIVE CORRELATION: Model performs BETTER on smaller deposits")
        print("  → Unexpected result - investigate feature patterns")
        print("  → Small deposits may have more distinctive signatures")
    else:
        print("✓ NO SIGNIFICANT CORRELATION: Model performance is UNIFORM across deposit sizes")
        print("  → Good balance in detection capability")
        print("  → Tonnage weighting does not bias towards large deposits")
    
    print(f"\nRecommendation:")
    print(f"  - If bias towards large deposits exists, consider balanced sampling")
    print(f"  - For exploration, prioritize high-confidence predictions of ANY size")
    print(f"  - Validate that small deposits aren't being systematically missed")


In [ ]:
# ==============================================================================
# STEP 7.6: FEATURE IMPORTANCE COMPARISON - WEIGHTED VS UNWEIGHTED
# ==============================================================================
print("\n" + "="*80)
print("STEP 7.6: Compare Feature Importance WITH and WITHOUT Tonnage Weighting")
print("="*80)

print("\nThis analysis reveals:")
print("  1. Which features are emphasized by large deposits (tonnage-weighted model)")
print("  2. Which features are universal predictors (unweighted model)")
print("  3. Genetic differences in feature patterns between large and small deposits")

# Train UNWEIGHTED model for comparison
print("\n" + "-"*80)
print("Training UNWEIGHTED model (no tonnage influence)...")
print("-"*80)

# Use same base estimator configuration
# base_estimator_unweighted = xgb.XGBClassifier(
#     n_estimators=100,
#     max_depth=best_params['max_depth'],
#     learning_rate=best_params['learning_rate'],
#     min_child_weight=best_params['min_child_weight'],
#     subsample=best_params['subsample'],
#     colsample_bytree=best_params['colsample_bytree'],
#     gamma=0.1,
#     reg_alpha=best_params['reg_alpha'],
#     reg_lambda=best_params['reg_lambda'],
#     random_state=RANDOM_STATE,
#     n_jobs=1,
#     eval_metric='logloss',
#     use_label_encoder=False
# )


base_estimator_unweighted  = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=1,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,                # Fixed: minimum loss reduction for split
    reg_alpha=0,
    reg_lambda=0.5,
    random_state=RANDOM_STATE,
    n_jobs=1,                 # Parallelism handled at bag level
    eval_metric='logloss',
    use_label_encoder=False
)

pu_model_unweighted = PUBaggingClassifier(
    base_estimator=base_estimator_unweighted,
    n_bags=100,
    bag_size=0.7,
    positive_label=1,
    unlabeled_label=0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

# Train WITHOUT sample weights (uniform weights)
print("Training with uniform weights (all deposits equal)...")
# Explicitly use uniform weights (all 1.0) - no tonnage influence
uniform_weights = np.ones(len(y_train))
pu_model_unweighted.fit(X_train, y_train, sample_weight=uniform_weights)
print("✓ Unweighted model training complete")
print(f"  Note: All samples weighted equally (weight=1.0)")

# Get feature importances from both models
importance_weighted_mean, importance_weighted_std = pu_model.get_feature_importances()
importance_unweighted_mean, importance_unweighted_std = pu_model_unweighted.get_feature_importances()

# Create comparison DataFrame
importance_comparison = pd.DataFrame({
    'feature': X_clean.columns,
    'importance_weighted': importance_weighted_mean,
    'importance_unweighted': importance_unweighted_mean,
    'difference': importance_weighted_mean - importance_unweighted_mean,
    'ratio': importance_weighted_mean / (importance_unweighted_mean + 1e-10),
})

# Assign geological groups
group_mapping = {}
for feature in X_clean.columns:
    assigned = False
    for group_name, patterns in feature_groups_patterns.items():
        for pattern in patterns:
            regex_pattern = pattern.replace('*', '.*')
            if re.match(f'^{regex_pattern}$', feature, re.IGNORECASE):
                group_mapping[feature] = group_name
                assigned = True
                break
        if assigned:
            break
    if not assigned:
        group_mapping[feature] = 'ungrouped'

importance_comparison['group'] = importance_comparison['feature'].map(group_mapping)

# Sort by absolute difference
importance_comparison['abs_difference'] = np.abs(importance_comparison['difference'])
importance_comparison_sorted = importance_comparison.sort_values('abs_difference', ascending=False)

print(f"\n{'='*80}")
print("FEATURE IMPORTANCE COMPARISON RESULTS")
print(f"{'='*80}")

print("\nTop 15 features EMPHASIZED by tonnage weighting (large deposits):")
print("(Positive difference = more important in weighted model)")
top_weighted = importance_comparison_sorted[importance_comparison_sorted['difference'] > 0].head(15)
print(top_weighted[['feature', 'importance_weighted', 'importance_unweighted', 'difference', 'group']].to_string(index=False))

print("\n\nTop 15 features DE-EMPHASIZED by tonnage weighting:")
print("(Negative difference = more important in unweighted model)")
top_unweighted = importance_comparison_sorted[importance_comparison_sorted['difference'] < 0].head(15)
print(top_unweighted[['feature', 'importance_weighted', 'importance_unweighted', 'difference', 'group']].to_string(index=False))

# Group-level comparison
print("\n\nGeological Group-Level Comparison:")
group_comparison = importance_comparison.groupby('group').agg({
    'importance_weighted': 'sum',
    'importance_unweighted': 'sum',
    'difference': 'sum'
}).sort_values('difference', ascending=False)

print(group_comparison.to_string())

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. Scatter: Weighted vs Unweighted importance
ax = axes[0, 0]
scatter = ax.scatter(importance_comparison['importance_unweighted'], 
                    importance_comparison['importance_weighted'],
                    c=importance_comparison['abs_difference'], 
                    cmap='coolwarm', s=30, alpha=0.6, edgecolors='black', linewidth=0.3)

# Add diagonal line (y=x)
max_val = max(importance_comparison['importance_weighted'].max(), 
              importance_comparison['importance_unweighted'].max())
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=2, alpha=0.5, label='Equal Importance')

# Annotate top different features
top_diff_features = importance_comparison_sorted.head(10)
for _, row in top_diff_features.iterrows():
    ax.annotate(row['feature'][:20], 
               (row['importance_unweighted'], row['importance_weighted']),
               fontsize=7, alpha=0.7)

ax.set_xlabel('Importance (Unweighted Model)', fontsize=11)
ax.set_ylabel('Importance (Weighted Model)', fontsize=11)
ax.set_title('Feature Importance: Weighted vs Unweighted', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Absolute Difference')

# 2. Top features emphasized by tonnage
ax = axes[0, 1]
top_n = 20
top_emphasized = importance_comparison_sorted[importance_comparison_sorted['difference'] > 0].head(top_n)
y_pos = np.arange(len(top_emphasized))
bars = ax.barh(y_pos, top_emphasized['difference'], color='coral', 
              edgecolor='black', linewidth=0.5, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_emphasized['feature'], fontsize=7)
ax.set_xlabel('Importance Increase (Weighted - Unweighted)', fontsize=11)
ax.set_title(f'Top {top_n} Features Emphasized by Large Deposits', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# 3. Top features de-emphasized by tonnage
ax = axes[1, 0]
top_deemphasized = importance_comparison_sorted[importance_comparison_sorted['difference'] < 0].head(top_n)
y_pos = np.arange(len(top_deemphasized))
bars = ax.barh(y_pos, np.abs(top_deemphasized['difference']), color='skyblue', 
              edgecolor='black', linewidth=0.5, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_deemphasized['feature'], fontsize=7)
ax.set_xlabel('Importance Decrease (|Weighted - Unweighted|)', fontsize=11)
ax.set_title(f'Top {top_n} Features More Important for Small Deposits', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# 4. Group-level comparison
ax = axes[1, 1]
groups = group_comparison.index
x_pos = np.arange(len(groups))
width = 0.35

bars1 = ax.bar(x_pos - width/2, group_comparison['importance_weighted'], width, 
              label='Weighted (Tonnage)', color='coral', edgecolor='black', linewidth=1, alpha=0.8)
bars2 = ax.bar(x_pos + width/2, group_comparison['importance_unweighted'], width, 
              label='Unweighted (Uniform)', color='skyblue', edgecolor='black', linewidth=1, alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(groups, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Total Importance', fontsize=11)
ax.set_title('Geological Group Importance: Weighted vs Unweighted', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add difference annotations
for i, (g, diff) in enumerate(zip(groups, group_comparison['difference'])):
    if diff > 0:
        ax.text(i, group_comparison.loc[g, 'importance_weighted'] + 0.01, 
               f'+{diff:.3f}', ha='center', va='bottom', fontsize=8, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('feature_importance_comparison_weighted_vs_unweighted.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Feature importance comparison plots saved to: feature_importance_comparison_weighted_vs_unweighted.png")
plt.show()

# Save comparison results
importance_comparison_sorted.to_csv('feature_importance_comparison.csv', index=False)
print(f"✓ Feature importance comparison saved to: feature_importance_comparison.csv")

group_comparison.to_csv('group_importance_comparison.csv')
print(f"✓ Group importance comparison saved to: group_importance_comparison.csv")

# Interpretation
print(f"\n{'='*80}")
print("INTERPRETATION & GEOLOGICAL INSIGHTS:")
print(f"{'='*80}")

# Identify which groups are emphasized
most_emphasized_group = group_comparison['difference'].idxmax()
most_deemphasized_group = group_comparison['difference'].idxmin()

print(f"\nMost emphasized by large deposits: {most_emphasized_group}")
print(f"  → Large deposits show stronger signals in {most_emphasized_group} features")
print(f"  → These may represent more intense mineralization signatures")

print(f"\nMost emphasized by small deposits: {most_deemphasized_group}")
print(f"  → Small deposits rely more on {most_deemphasized_group} features")
print(f"  → These may be more subtle or specific indicators")

print(f"\nKey findings:")
print(f"  1. Features with positive difference → more predictive of LARGE deposits")
print(f"  2. Features with negative difference → more predictive of SMALL deposits")
print(f"  3. Features near diagonal → universal predictors (all sizes)")

print(f"\nRecommendations:")
print(f"  - Use WEIGHTED model for targeting large economic deposits")
print(f"  - Use UNWEIGHTED model for comprehensive deposit discovery")
print(f"  - Combine both models for balanced exploration strategy")
print(f"  - Investigate top differential features for genetic understanding")

# Evaluate both models on test set
y_pred_unweighted, _ = pu_model_unweighted.predict_with_uncertainty(X_test)
auc_roc_unweighted = roc_auc_score(y_test, y_pred_unweighted)
auc_pr_unweighted = average_precision_score(y_test, y_pred_unweighted)

print(f"\n{'='*80}")
print("MODEL PERFORMANCE COMPARISON:")
print(f"{'='*80}")
print(f"Weighted Model (Tonnage-aware):")
print(f"  AUC-ROC: {auc_roc:.4f}")
print(f"  AUC-PR:  {auc_pr:.4f}")
print(f"\nUnweighted Model (Uniform):")
print(f"  AUC-ROC: {auc_roc_unweighted:.4f}")
print(f"  AUC-PR:  {auc_pr_unweighted:.4f}")
print(f"\nDifference:")
print(f"  ΔAUC-ROC: {auc_roc - auc_roc_unweighted:+.4f}")
print(f"  ΔAUC-PR:  {auc_pr - auc_pr_unweighted:+.4f}")

if auc_roc > auc_roc_unweighted:
    print(f"\n✓ Tonnage weighting IMPROVES overall performance")
else:
    print(f"\n⚠ Tonnage weighting does NOT improve overall performance")
    print(f"  Consider using unweighted model or investigating feature engineering")


In [ ]:
# ==============================================================================
# STEP 8: SUMMARY AND RECOMMENDATIONS
# ==============================================================================
print("\n" + "="*80)
print("NORTH AMERICAN COPPER PROSPECTIVITY - WORKFLOW SUMMARY")
print("="*80)

print(f"""
COMPLETED WORKFLOW STEPS:
✓ 1. Loaded real copper prospectivity dataset ({len(y):,} samples)
✓ 2. Applied geology-aware feature hygiene
     - Original features: {X_df.shape[1]}
     - Retained features: {X_clean.shape[1]}
     - Reduction: {100*(1 - X_clean.shape[1]/X_df.shape[1]):.1f}%
✓ 3. Split data with stratification
✓ 3.5. Optimized hyperparameters via Grid Search CV
     - Best CV AUC-ROC: {random_search.best_score_:.4f}
     - Parameters tuned: max_depth, learning_rate, subsample, etc.
✓ 4. Trained PU-bagging classifier with {pu_model.n_bags} bags (optimized params)
✓ 5. Generated predictions with uncertainty quantification
✓ 6. Analyzed feature importances
✓ 7. Performed {'SHAP-based' if shap_success else 'basic'} diagnostics

MODEL PERFORMANCE:
- AUC-ROC: {auc_roc:.3f}
- AUC-PR: {auc_pr:.3f}
- Mean prediction uncertainty: {y_pred_std.mean():.3f}
- Positive samples: {np.sum(y==1):,}
- Unlabeled samples: {np.sum(y==0):,}

FEATURE INSIGHTS:
- Retained {X_clean.shape[1]} of {X_df.shape[1]} original features
- Top geological group: {importance_df.groupby('group')['importance_mean'].sum().idxmax()}
- Features with high selection frequency: {(selection_freq > 0.9).sum()}
- Top 3 most important features:
  1. {importance_df.iloc[0]['feature']} (importance: {importance_df.iloc[0]['importance_mean']:.4f})
  2. {importance_df.iloc[1]['feature']} (importance: {importance_df.iloc[1]['importance_mean']:.4f})
  3. {importance_df.iloc[2]['feature']} (importance: {importance_df.iloc[2]['importance_mean']:.4f})

WEIGHTS USED:
- Sample weights incorporated from tonnage_mt
- Mean weight for positive samples: {weights_train[y_train==1].mean():.2f}
- This accounts for deposit size in model training

OUTPUT FILES GENERATED:
✓ feature_removal_report.csv - Details of removed features
✓ hyperparameter_tuning_results.csv - Grid search CV results
✓ hyperparameter_tuning_analysis.png - Hyperparameter effects visualization
✓ feature_importance_report.csv - Complete importance rankings
✓ pu_predictions_copper.png - Prediction distribution plots
✓ feature_importance_copper.png - Importance visualizations
{"✓ feature_stability_report_copper.csv - SHAP stability analysis" if shap_success else ""}
{"✓ feature_stability_copper.png - Stability diagnostic plots" if shap_success else ""}

FOR PUBLICATION:
1. Report all hyperparameters AND the tuning process (Grid Search CV with {total_combinations} combinations)
2. Include best CV score: {random_search.best_score_:.4f}
3. Report optimized parameters: max_depth={best_params['max_depth']}, learning_rate={best_params['learning_rate']}, etc.
4. Mention n_bags={pu_model.n_bags}, bag_size={pu_model.bag_size}
2. Include uncertainty quantification in prospectivity maps
3. Discuss geological meaning of top features (see importance report)
4. Validate on held-out deposits or spatial cross-validation
5. Compare against standard supervised learning baseline
6. Include SHAP plots for interpretability (if available)
7. Discuss unstable features and removal rationale

NEXT STEPS FOR SPATIAL PROSPECTIVITY MAPPING:
1. Generate prospectivity maps using Latitude/Longitude
2. Create uncertainty maps (y_pred_std) to identify areas needing more data
3. Implement spatial cross-validation to avoid spatial autocorrelation
4. Validate predictions against new discoveries
5. Compare with existing prospectivity maps
6. Identify high-potential targets for exploration

GEOLOGICAL INTERPRETATION PRIORITY:
- Examine top features for geological consistency
- Check if alteration signatures align with deposit type
- Verify geophysical anomalies match expected signatures
- Assess structural control importance
- Consider regional vs. local-scale features
""")

print("="*80)
print("Workflow complete! All outputs saved to current directory.")
print("="*80)

# Create summary statistics dictionary for easy access
workflow_summary = {
    'n_samples_total': len(y),
    'n_positive': np.sum(y==1),
    'n_unlabeled': np.sum(y==0),
    'n_features_original': X_df.shape[1],
    'n_features_retained': X_clean.shape[1],
    'feature_reduction_pct': 100*(1 - X_clean.shape[1]/X_df.shape[1]),
    'auc_roc': auc_roc,
    'auc_pr': auc_pr,
    'mean_prediction_uncertainty': y_pred_std.mean(),
    'top_geological_group': importance_df.groupby('group')['importance_mean'].sum().idxmax(),
    'n_bags': pu_model.n_bags,
    'bag_size': pu_model.bag_size,
    'best_cv_score': random_search.best_score_,
    'best_params': best_params,
    'n_hyperparameter_combinations_tested': total_combinations,
    'random_state': RANDOM_STATE
}

print("\n📊 Summary statistics stored in 'workflow_summary' dictionary")

In [ ]:
# ==============================================================================
# OPTIONAL: SPATIAL VISUALIZATION OF PREDICTIONS
# ==============================================================================
print("\n" + "="*80)
print("OPTIONAL: Spatial Visualization of Test Predictions")
print("="*80)

# Get latitude/longitude for test samples
test_indices = X_test.index
test_locations = ALL_DATA.loc[test_indices, ['Latitude', 'Longitude']].copy()
test_locations['prediction'] = y_pred_mean
test_locations['uncertainty'] = y_pred_std
test_locations['true_label'] = y_test

# Create spatial visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Prospectivity map
ax = axes[0]
scatter = ax.scatter(test_locations['Longitude'], test_locations['Latitude'], 
                    c=test_locations['prediction'], cmap='YlOrRd', 
                    s=10, alpha=0.6, edgecolors='none')
# Overlay known deposits
deposits = test_locations[test_locations['true_label'] == 1]
ax.scatter(deposits['Longitude'], deposits['Latitude'], 
          marker='^', s=100, c='blue', edgecolors='white', 
          linewidths=1, label='Known Deposits', zorder=5)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('Prospectivity Map (Test Set)', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax, label='Prospectivity Score')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Uncertainty map
ax = axes[1]
scatter = ax.scatter(test_locations['Longitude'], test_locations['Latitude'], 
                    c=test_locations['uncertainty'], cmap='viridis', 
                    s=10, alpha=0.6, edgecolors='none')
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('Prediction Uncertainty Map', fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax, label='Uncertainty (Std)')
ax.grid(True, alpha=0.3)

# 3. High-confidence prospectivity (low uncertainty, high prediction)
ax = axes[2]
# Define high-confidence areas: high prediction AND low uncertainty
high_conf_threshold_pred = 0.6
low_uncertainty_threshold = 0.15
high_confidence = test_locations[
    (test_locations['prediction'] > high_conf_threshold_pred) & 
    (test_locations['uncertainty'] < low_uncertainty_threshold)
]

# Plot all points in gray
ax.scatter(test_locations['Longitude'], test_locations['Latitude'], 
          c='lightgray', s=5, alpha=0.3, edgecolors='none')
# Highlight high-confidence prospective areas
ax.scatter(high_confidence['Longitude'], high_confidence['Latitude'], 
          c=high_confidence['prediction'], cmap='YlOrRd', 
          s=30, alpha=0.8, edgecolors='black', linewidths=0.3)
# Overlay known deposits
ax.scatter(deposits['Longitude'], deposits['Latitude'], 
          marker='^', s=100, c='blue', edgecolors='white', 
          linewidths=1, label='Known Deposits', zorder=5)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title(f'High-Confidence Targets (n={len(high_confidence)})', 
            fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('spatial_prospectivity_maps_copper.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Spatial maps saved to: spatial_prospectivity_maps_copper.png")
plt.show()

print(f"\nHigh-confidence target statistics:")
print(f"  Total high-confidence areas: {len(high_confidence)}")
print(f"  Mean prospectivity: {high_confidence['prediction'].mean():.3f}")
print(f"  Mean uncertainty: {high_confidence['uncertainty'].mean():.3f}")
print(f"  Contains {(high_confidence['true_label'] == 1).sum()} known deposits")

# Save high-confidence targets for further analysis
high_confidence.to_csv('high_confidence_targets.csv', index=False)
print(f"✓ High-confidence targets saved to: high_confidence_targets.csv")

# Summary statistics
print(f"\nSpatial coverage (test set):")
print(f"  Longitude range: {test_locations['Longitude'].min():.2f} to {test_locations['Longitude'].max():.2f}")
print(f"  Latitude range: {test_locations['Latitude'].min():.2f} to {test_locations['Latitude'].max():.2f}")

In [ ]:
# ==============================================================================
# GRID PREDICTION: FULL REGIONAL PROSPECTIVITY MAP
# ==============================================================================
print("\n" + "="*80)
print("GRID PREDICTION: Generate Regional Prospectivity Map")
print("="*80)

# Load regular grid data
grid_path = "<DATA_ROOT>/Paper/Zenodo_DataBundle/grid_data/Regular_Grid_Sampled.csv"
print(f"\nLoading grid data from: {grid_path}")
test_df = pd.read_csv(grid_path)

print(f"Grid data shape: {test_df.shape}")
print(f"Total grid points: {len(test_df):,}")


In [ ]:

# Extract coordinates (save for mapping)
if 'Latitude' in test_df.columns and 'Longitude' in test_df.columns:
    grid_coords = test_df[['Latitude', 'Longitude']].copy()
    print(f"\nGrid spatial coverage:")
    print(f"  Latitude: {grid_coords['Latitude'].min():.2f} to {grid_coords['Latitude'].max():.2f}")
    print(f"  Longitude: {grid_coords['Longitude'].min():.2f} to {grid_coords['Longitude'].max():.2f}")
else:
    print("\n⚠ Warning: Latitude/Longitude not found in grid data")
    grid_coords = None

# Extract features (match training feature set)
grid_feature_cols = [col for col in feature_cols if col in test_df.columns]
missing_features = set(feature_cols) - set(grid_feature_cols)

if len(missing_features) > 0:
    print(f"\n⚠ Warning: {len(missing_features)} features missing from grid data")
    print(f"  Missing features will be filled with 0")
    # Create full feature matrix with zeros for missing features
    X_grid_full = pd.DataFrame(0, index=test_df.index, columns=feature_cols)
    X_grid_full[grid_feature_cols] = test_df[grid_feature_cols]
    X_grid = X_grid_full
else:
    X_grid = test_df[feature_cols].copy()

print(f"\nFeature extraction complete:")
print(f"  Grid features: {X_grid.shape[1]}")
print(f"  Matching training features: {len(grid_feature_cols)}")

# Apply same feature selection as training
print("\nApplying feature selection (same as training)...")
X_grid_clean = feature_selector.transform(X_grid)
print(f"  Grid features after selection: {X_grid_clean.shape[1]}")
print(f"  Expected features: {X_clean.shape[1]}")

if X_grid_clean.shape[1] != X_clean.shape[1]:
    print(f"\n⚠ Warning: Feature count mismatch!")
else:
    print(f"  ✓ Feature dimensions match")

# Predict prospectivity with uncertainty
print("\nGenerating predictions for grid...")
print("  This may take a few minutes for large grids...")

grid_pred_mean, grid_pred_std = pu_model.predict_with_uncertainty(X_grid_clean)

print(f"\n✓ Grid prediction complete")
print(f"  Mean prospectivity: {grid_pred_mean.mean():.3f}")
print(f"  Std prospectivity: {grid_pred_mean.std():.3f}")
print(f"  Min: {grid_pred_mean.min():.3f}, Max: {grid_pred_mean.max():.3f}")
print(f"  Mean uncertainty: {grid_pred_std.mean():.3f}")

# Create results dataframe
grid_results = pd.DataFrame({
    'prospectivity': grid_pred_mean,
    'uncertainty': grid_pred_std
})

if grid_coords is not None:
    grid_results['Latitude'] = grid_coords['Latitude'].values
    grid_results['Longitude'] = grid_coords['Longitude'].values

# Save results
grid_results.to_csv('grid_prospectivity_predictions.csv', index=False)
print(f"\n✓ Grid predictions saved to: grid_prospectivity_predictions.csv")


In [ ]:
grid_results.head()

In [ ]:
from pyDTDM.utils import *
nc=df_to_NetCDF(grid_results['Longitude'], grid_results['Latitude'], grid_results['prospectivity'],grid_resolution=0.05)
nc.plot()

In [ ]:
nc.to_netcdf('spatial_grid_predictions.nc')

In [ ]:

# Generate regional prospectivity map
if grid_coords is not None:
    print("\nGenerating regional prospectivity maps...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # 1. Prospectivity map
    ax = axes[0, 0]
    scatter = ax.scatter(grid_results['Longitude'], grid_results['Latitude'], 
                        c=grid_results['prospectivity'], cmap='YlOrRd', 
                        s=5, alpha=0.7, edgecolors='none', vmin=0, vmax=1)
    # Overlay known deposits from training data
    deposits = ALL_DATA[ALL_DATA['label_binary'] == 1]
    ax.scatter(deposits['Longitude'], deposits['Latitude'], 
              marker='*', s=150, c='blue', edgecolors='white', 
              linewidths=1.5, label=f'Known Deposits (n={len(deposits)})', zorder=5)
    ax.set_xlabel('Longitude', fontsize=11)
    ax.set_ylabel('Latitude', fontsize=11)
    ax.set_title('Regional Copper Prospectivity Map', fontsize=12, fontweight='bold')
    cbar = plt.colorbar(scatter, ax=ax, label='Prospectivity Score')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.2)
    
    # 2. Uncertainty map
    ax = axes[0, 1]
    scatter = ax.scatter(grid_results['Longitude'], grid_results['Latitude'], 
                        c=grid_results['uncertainty'], cmap='viridis', 
                        s=5, alpha=0.7, edgecolors='none')
    ax.set_xlabel('Longitude', fontsize=11)
    ax.set_ylabel('Latitude', fontsize=11)
    ax.set_title('Prediction Uncertainty Map', fontsize=12, fontweight='bold')
    cbar = plt.colorbar(scatter, ax=ax, label='Uncertainty (Std)')
    ax.grid(True, alpha=0.2)
    
    # 3. High prospectivity areas (top 5%)
    ax = axes[1, 0]
    threshold_95 = np.percentile(grid_results['prospectivity'], 95)
    high_prosp = grid_results[grid_results['prospectivity'] >= threshold_95]
    
    ax.scatter(grid_results['Longitude'], grid_results['Latitude'], 
              c='lightgray', s=2, alpha=0.3, edgecolors='none')
    scatter = ax.scatter(high_prosp['Longitude'], high_prosp['Latitude'], 
                        c=high_prosp['prospectivity'], cmap='YlOrRd', 
                        s=20, alpha=0.8, #edgecolors='black', 
                        linewidths=0.3, vmin=threshold_95, vmax=1)
    ax.scatter(deposits['Longitude'], deposits['Latitude'], 
              marker='*', s=150, c='blue', edgecolors='white', 
              linewidths=1.5, label='Known Deposits', zorder=5,alpha=0.1)
    ax.set_xlabel('Longitude', fontsize=11)
    ax.set_ylabel('Latitude', fontsize=11)
    ax.set_title(f'High Prospectivity Areas (Top 5%, n={len(high_prosp):,})', 
                fontsize=12, fontweight='bold')
    cbar = plt.colorbar(scatter, ax=ax, label='Prospectivity Score')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.2)
    
    # 4. High-confidence targets (high prospectivity + low uncertainty)
    ax = axes[1, 1]
    threshold_prosp = np.percentile(grid_results['prospectivity'], 90)
    threshold_unc = np.percentile(grid_results['uncertainty'], 50)  # Below median uncertainty
    
    high_confidence_grid = grid_results[
        (grid_results['prospectivity'] >= threshold_prosp) & 
        (grid_results['uncertainty'] <= threshold_unc)
    ]
    
    ax.scatter(grid_results['Longitude'], grid_results['Latitude'], 
              c='lightgray', s=2, alpha=0.3, edgecolors='none')
    scatter = ax.scatter(high_confidence_grid['Longitude'], high_confidence_grid['Latitude'], 
                        c=high_confidence_grid['prospectivity'], cmap='YlOrRd', 
                        s=20, alpha=0.8, #edgecolors='black',
                          linewidths=0.3)
    ax.scatter(deposits['Longitude'], deposits['Latitude'], 
              marker='*', s=150, c='blue', edgecolors='white', 
              linewidths=1.5, label='Known Deposits', zorder=5,alpha=0.1)
    ax.set_xlabel('Longitude', fontsize=11)
    ax.set_ylabel('Latitude', fontsize=11)
    ax.set_title(f'High-Confidence Exploration Targets (n={len(high_confidence_grid):,})', 
                fontsize=12, fontweight='bold')
    cbar = plt.colorbar(scatter, ax=ax, label='Prospectivity Score')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.savefig('regional_prospectivity_map.png', dpi=300, bbox_inches='tight')
    print(f"✓ Regional prospectivity maps saved to: regional_prospectivity_map.png")
    plt.show()
    
    # Save high-confidence targets
    high_confidence_grid.to_csv('high_confidence_grid_targets.csv', index=False)
    print(f"\n✓ High-confidence targets saved to: high_confidence_grid_targets.csv")
    
    print(f"\nExploration targeting summary:")
    print(f"  Total grid points: {len(grid_results):,}")
    print(f"  High prospectivity (top 5%): {len(high_prosp):,}")
    print(f"  High-confidence targets: {len(high_confidence_grid):,}")
    print(f"  Known deposits in region: {len(deposits):,}")
    print(f"  \nRecommendation: Prioritize high-confidence targets for field validation")

print("\n" + "="*80)
print("GRID PREDICTION COMPLETE")
print("="*80)

In [ ]:
import os
import pyproj
# Point pyproj to the correct PROJ data directory
os.environ["PROJ_LIB"] = "<CONDA>/envs/EBMTest311/share/proj"
pyproj.datadir.set_data_dir(os.environ["PROJ_LIB"])
coastlines_file="<DATA_ROOT>/Raw/plate_model/StaticGeometries/Coastlines/Global_coastlines_low_res.shp"
coastlines_gdf=gpd.read_file(coastlines_file)


In [ ]:





# Generate regional prospectivity map with deposit size visualization
if grid_coords is not None:
    print("\nGenerating regional prospectivity maps with deposit tonnage sizing...")
    
    # Prepare deposit data with tonnage information
    deposits = ALL_DATA[ALL_DATA['label_binary'] == 1].copy()
    
    # Fill NaN tonnage values with 1 (as requested)
    deposits['tonnage_for_plot'] = deposits['tonnage_mt']
   #  .fillna(1)
    
    # Scale marker sizes: logarithmic transform for better visual spread
    # Base size 80, scaled by log of tonnage
    marker_sizes = 25 * np.log10(deposits['tonnage_for_plot'] + 1)
    
    print(f"\nDeposit tonnage statistics:")
    print(f"  Total deposits: {len(deposits)}")
    print(f"  Deposits with tonnage data: {deposits['tonnage_mt'].notna().sum()}")
    print(f"  Deposits with NaN (set to 1): {deposits['tonnage_mt'].isna().sum()}")
    print(f"  Tonnage range: {deposits['tonnage_for_plot'].min():.2f} - {deposits['tonnage_for_plot'].max():.2f} Mt")
    print(f"  Mean tonnage: {deposits['tonnage_for_plot'].mean():.2f} Mt")
    print(f"  Median tonnage: {deposits['tonnage_for_plot'].median():.2f} Mt")
    
    fig_scale = 0.8
    # 2-row figure
    fig, axes = plt.subplots(2, 1, dpi=300, figsize=(10*fig_scale, 16*fig_scale))
    # 1. Regional Copper Prospectivity Map
    ax = axes[0]

     
    coastlines_gdf.plot(ax=ax, 
                        # facecolor="None", 
                        edgecolor="grey", alpha=0.1)
    
    scatter = ax.scatter(grid_results['Longitude'], grid_results['Latitude'], 
                        c=grid_results['prospectivity'], cmap='YlOrRd', 
                        s=5, alpha=0.7, edgecolors='none', vmin=0, vmax=1)
    
    # Overlay known deposits with single color, sized by tonnage
    ax.scatter(deposits['Longitude'], deposits['Latitude'], 
               s=marker_sizes,
            #    c='cyan',  # Single high-contrast color
            facecolor="None",
            #    marker='*', 
               edgecolors='black', 
               linewidths=1.2, 
               alpha=0.3,
               zorder=5)
   # Turn off x-axis label and ticks
    ax.set_xlabel("Longitude")
    ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=True)

    ax.set_ylabel('Latitude', fontsize=12)
    ax.set_title('Spatial Prospectivity Map', fontsize=13, fontweight='bold')

    ax.set_xlim(-145, -100)
    ax.set_ylim(25, 72)
      
    # Colorbar for prospectivity only
    cbar_prosp = plt.colorbar(scatter, ax=ax, label='Spatial Prospectivity Score')
    
    # Create simplified legend for deposit sizes
    size_legend_elements = [
        plt.scatter([], [], s=25*np.log10(1+1), 
                    # marker='*', 
                    # c='cyan', 
                    facecolor='None',
                   edgecolors='black', 
                   linewidths=1.2, label='1 Mt'),
        plt.scatter([], [], s=25*np.log10(10+1), 
                    # marker='*', 
                    # c='cyan', 
                    facecolor='None',
                   edgecolors='black', linewidths=1.2, label='10 Mt'),
        plt.scatter([], [], s=25*np.log10(100+1), 
                    # marker='*', 
                    # c='cyan', 
                    facecolor='None',
                   edgecolors='black', linewidths=1.2, label='100 Mt'),
    ]
    legend1 = ax.legend(handles=size_legend_elements, 
                       loc='upper right', fontsize=10, 
                       title=f'Known Deposits',
                       title_fontsize=11,
                       framealpha=0.95)
   #  ax.grid(True, alpha=0.2)
    
    # 2. High Prospectivity Areas (top 5%)
    ax = axes[1]
    threshold_95 = np.percentile(grid_results['prospectivity'], 95)
    high_prosp = grid_results[grid_results['prospectivity'] >= threshold_95]

    coastlines_gdf.plot(ax=ax, 
                    # facecolor="None", 
                    edgecolor="grey", alpha=0.1)
    
    # Background grid in light gray
    ax.scatter(grid_results['Longitude'], grid_results['Latitude'], 
              c='lightgray', s=2, alpha=0.3, edgecolors='none')
    
    # High prospectivity areas
    scatter = ax.scatter(high_prosp['Longitude'], high_prosp['Latitude'], 
                        c=high_prosp['prospectivity'], cmap='inferno_r', 
                        s=20, alpha=0.7,
                        edgecolor="None",
                        # linewidths=0.3,
                          vmin=threshold_95, vmax=1)
    
    # Overlay known deposits with single color, sized by tonnage
    ax.scatter(deposits['Longitude'], deposits['Latitude'], 
               s=marker_sizes,
            #    c='cyan',  # Single high-contrast color
               facecolor='None',
            #    marker='*', 
               edgecolors='black', 
               linewidths=1.2, 
               alpha=0.05,
               zorder=5)
    

    
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.set_title(f'High Prospectivity Areas (Top 5%)', 
                fontsize=13, fontweight='bold')
    
    # Colorbar for prospectivity only
    cbar_prosp = plt.colorbar(scatter, ax=ax, label='Spatial Prospectivity Score')
    
    # Legend for deposit sizes
    # legend1 = ax.legend(handles=size_legend_elements, 
    #                    loc='upper right', fontsize=10, 
    #                    title=f'Known Deposits (n={len(deposits)})',
    #                    title_fontsize=11,
    #                    framealpha=0.95)
   #  ax.grid(True, alpha=0.2)
    ax.set_xlim(-145, -100)
    ax.set_ylim(25, 72)
    plt.tight_layout()
    plt.savefig('regional_prospectivity_with_tonnage.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Regional prospectivity maps with tonnage saved to: regional_prospectivity_with_tonnage.png")
    plt.show()
    
    # Summary statistics
    print(f"\nExploration targeting summary:")
    print(f"  Total grid points: {len(grid_results):,}")
    print(f"  High prospectivity (top 5%): {len(high_prosp):,}")
    print(f"  Known deposits in region: {len(deposits):,}")
    
    # Identify deposits in high prospectivity areas
    from scipy.spatial.distance import cdist
    
    # Find deposits within high prospectivity regions (using nearest neighbor)
    if len(high_prosp) > 0 and len(deposits) > 0:
        deposit_coords = deposits[['Longitude', 'Latitude']].values
        high_prosp_coords = high_prosp[['Longitude', 'Latitude']].values
        
        # Calculate minimum distance from each deposit to high prospectivity areas
        distances = cdist(deposit_coords, high_prosp_coords)
        min_distances = distances.min(axis=1)
        
        # Deposits within ~0.5 degree (~55km) of high prospectivity areas
        nearby_threshold = 0.5
        deposits_in_high_prosp = deposits[min_distances <= nearby_threshold]
        
        print(f"  Deposits within {nearby_threshold}° of high prospectivity areas: {len(deposits_in_high_prosp)}")
        print(f"  Total tonnage in high prospectivity regions: {deposits_in_high_prosp['tonnage_for_plot'].sum():.2f} Mt")
    
    print(f"\n  Recommendation: Prioritize high prospectivity areas for exploration")

print("\n" + "="*80)
print("GRID PREDICTION COMPLETE")
print("="*80)


In [ ]:
# ==============================================================================
# GRID PREDICTION: FULL REGIONAL PROSPECTIVITY MAP
# ==============================================================================
print("\n" + "="*80)
print("GRID PREDICTION: Generate Regional Prospectivity Map")
print("="*80)

# Load regular grid data
grid_path = "<DATA_ROOT>/Raw/Vectors/Outputs/DatasetPU_USA/Regular_Grid_Sampled.csv"
print(f"\nLoading grid data from: {grid_path}")
test_df = pd.read_csv(grid_path)

print(f"Grid data shape: {test_df.shape}")
print(f"Total grid points: {len(test_df):,}")

In [ ]:
test_df['Latitude'].max(), test_df['Latitude'].min(), test_df['Longitude'].max(), test_df['Longitude'].min()

In [ ]:
columns=["al2o3_laterite_argilic",
"ferrous_iron",
"al2o3_alteration",
"hydrothermal_alteration",
"sio2_silica_index",
"ferric_iron",
"GeophysicsGravity",
"GeophysicsGravity_HGM",
"GeophysicsGravity_Up30km",
"GeophysicsGravity_Up30km_HGM",
"GeophysicsMag",
"GeophysicsMag_RTP_VD",
"GeophysicsMag_RTP_HGM",
"GeophysicsMag_RTP_USCanada",
"Geologic_contacts_nearest_distance",
"Faults_nearest_distance",
"Faults_Compressional_nearest_distance",
"Faults_Extensional_nearest_distance",
"Faults_Strike_Slip_nearest_distance",
"Litho_Intrusive_Alkaline_nearest_distance",
"Litho_Intrusive_Felsic_nearest_distance",
"Litho_Mafic_Ultramafic_nearest_distance",
"Litho_Metamorphic_nearest_distance",
"Litho_Reactive_Host_nearest_distance",
]

In [ ]:
# ==============================================================================
# AUTOMATED SPATIAL FEATURE VISUALIZATION
# ==============================================================================
print("\n" + "="*80)
print("AUTOMATED SPATIAL FEATURE MAPPING")
print("="*80)

import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def plot_geospatial_features(df, output_folder='spatial_feature_maps', 
                             exclude_cols=None, include_cols=None,
                             figsize=(10, 8), dpi=300):
    """
    Automatically plot all numeric features as spatial maps with intelligent colormap selection.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Dataframe with 'Longitude', 'Latitude' columns and numeric feature columns
    output_folder : str
        Folder to save figures
    exclude_cols : list, optional
        List of column names to exclude from plotting
    include_cols : list, optional
        List of column names to explicitly include (if None, uses all numeric)
    figsize : tuple
        Figure size (width, height)
    dpi : int
        Resolution for saved figures
    """
    
    # Create output folder if it doesn't exist
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"\nOutput folder: {output_path.absolute()}")
    
    # Check required columns
    if 'Longitude' not in df.columns or 'Latitude' not in df.columns:
        print("ERROR: DataFrame must contain 'Longitude' and 'Latitude' columns")
        return
    
    # Define default columns to exclude
    default_exclude = ['Longitude', 'Latitude', 'label_binary', 'label', 
                      'sample_weight', 'tonnage_mt', 'tonnage_for_plot',
                      'index', 'id', 'ID', 'Index']
    
    if exclude_cols is not None:
        exclude_cols = list(set(default_exclude + exclude_cols))
    else:
        exclude_cols = default_exclude
    
    # Determine which columns to plot
    if include_cols is not None:
        # Use explicitly provided list
        plot_columns = [col for col in include_cols if col in df.columns]
    else:
        # Use all numeric columns except excluded ones
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        plot_columns = [col for col in numeric_cols if col not in exclude_cols]
    
    print(f"\nFound {len(plot_columns)} features to plot")
    print(f"Excluded columns: {exclude_cols}")
    
    if len(plot_columns) == 0:
        print("No columns to plot!")
        return
    
    # Publication quality settings
    plt.style.use('seaborn-v0_8-paper')
    plt.rcParams.update({
        'font.size': 10,
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'figure.titlesize': 13,
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans'],
    })
    
    # Get coordinates
    lon = df['Longitude'].values
    lat = df['Latitude'].values
    
    # Track statistics
    diverging_count = 0
    sequential_count = 0
    failed_plots = []
    
    print(f"\nStarting visualization...")
    print("-" * 80)
    
    # Iterate through each feature
    for idx, col in enumerate(plot_columns, 1):
        try:
            # Get data
            data = df[col].values
            
            # Skip if all NaN
            if np.all(np.isnan(data)):
                print(f"[{idx}/{len(plot_columns)}] SKIPPED: {col} (all NaN)")
                failed_plots.append((col, "all NaN"))
                continue
            
            # Remove NaN for statistics
            valid_mask = ~np.isnan(data)
            valid_data = data[valid_mask]
            
            if len(valid_data) == 0:
                print(f"[{idx}/{len(plot_columns)}] SKIPPED: {col} (no valid data)")
                failed_plots.append((col, "no valid data"))
                continue
            
            # Compute statistics
            vmin = np.nanmin(data)
            vmax = np.nanmax(data)
            vmean = np.nanmean(data)
            vmedian = np.nanmedian(data)
            vstd = np.nanstd(data)
            
            # Use percentiles for robust colormap normalization (handles outliers)
            # This prevents extreme values from compressing the color scale
            vmin_p2 = np.nanpercentile(valid_data, 2)   # 2nd percentile
            vmax_p98 = np.nanpercentile(valid_data, 98)  # 98th percentile
            
            # Determine colormap based on data distribution
            # If data spans negative and positive, use diverging
            # If data is strictly positive or strictly negative, use sequential
            if vmin_p2 < 0 and vmax_p98 > 0:
                # Diverging colormap centered at zero
                cmap = 'RdBu_r'
                diverging_count += 1
                cmap_type = 'diverging'
                
                # Make symmetric around zero for better visualization
                # Use percentiles to avoid outlier influence
                abs_max = max(abs(vmin_p2), abs(vmax_p98))
                norm_vmin = -abs_max
                norm_vmax = abs_max
            else:
                # Sequential colormap
                if 'distance' in col.lower():
                    cmap = 'viridis_r'  # Reverse for distances (low = close = interesting)
                elif any(x in col.lower() for x in ['gravity', 'mag', 'geophysics']):
                    cmap = 'plasma'
                elif any(x in col.lower() for x in ['iron', 'al2o3', 'sio2', 'alteration']):
                    cmap = 'YlOrRd'
                else:
                    cmap = 'viridis'
                
                sequential_count += 1
                cmap_type = 'sequential'
                # Use percentiles instead of min/max to handle outliers
                norm_vmin = vmin_p2
                norm_vmax = vmax_p98
            
            # Create figure
            fig, ax = plt.subplots(figsize=figsize)
            
            # Plot scatter
            scatter = ax.scatter(lon, lat, c=data, cmap=cmap, 
                               s=3, alpha=0.7, edgecolors='none',
                               vmin=norm_vmin, vmax=norm_vmax)
            
            # Colorbar
            cbar = plt.colorbar(scatter, ax=ax, label=col, pad=0.02)
            cbar.ax.tick_params(labelsize=9)
            
            # Labels and title
            ax.set_xlabel('Longitude', fontsize=11)
            ax.set_ylabel('Latitude', fontsize=11)
            ax.set_title(f'{col}', 
                        fontsize=12, fontweight='bold', pad=10)
            
            # Add statistics text box
            stats_text = (f'Min: {vmin:.3g}\n'
                         f'Max: {vmax:.3g}\n'
                         f'Mean: {vmean:.3g}\n'
                         f'Median: {vmedian:.3g}\n'
                         f'Std: {vstd:.3g}\n'
                         f'——————————\n'
                         f'Color scale (2-98%ile):\n'
                         f'[{norm_vmin:.3g}, {norm_vmax:.3g}]\n'
                         f'Valid: {len(valid_data):,}/{len(data):,}')
            
            ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
                   fontsize=7.5, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, 
                            edgecolor='gray', linewidth=0.5))
            
            # Grid
            ax.grid(True, alpha=0.2, linewidth=0.5)
            ax.set_aspect('auto')
            
            plt.tight_layout()
            
            # Save figure
            safe_filename = col.replace('/', '_').replace('\\', '_').replace(' ', '_')
            output_file = output_path / f'{safe_filename}.png'
            plt.savefig(output_file, dpi=dpi, bbox_inches='tight')
            plt.close()
            
            print(f"[{idx}/{len(plot_columns)}] ✓ {col:50s} | {cmap_type:10s} | range: [{vmin:.3g}, {vmax:.3g}]")
            
        except Exception as e:
            print(f"[{idx}/{len(plot_columns)}] ✗ ERROR: {col} - {str(e)}")
            failed_plots.append((col, str(e)))
            plt.close('all')
    
    # Summary
    print("\n" + "="*80)
    print("VISUALIZATION SUMMARY")
    print("="*80)
    print(f"Total features processed: {len(plot_columns)}")
    print(f"  Successfully plotted: {len(plot_columns) - len(failed_plots)}")
    print(f"    - Diverging colormaps: {diverging_count}")
    print(f"    - Sequential colormaps: {sequential_count}")
    print(f"  Failed/Skipped: {len(failed_plots)}")
    
    if failed_plots:
        print(f"\nFailed plots:")
        for col, reason in failed_plots:
            print(f"  - {col}: {reason}")
    
    print(f"\n✓ All plots saved to: {output_path.absolute()}")
    print("="*80)
    
    # Reset matplotlib style
    plt.style.use('default')
    
    return output_path


# Example usage with your grid data
# Uncomment and modify as needed:

# Option 1: Plot ALL numeric features from the grid data
print("\nOption 1: Plot all features from test_df (grid data)")
print("Uncomment the line below to execute:")
print("# output_folder = plot_geospatial_features(test_df, output_folder='grid_feature_maps')")

# Option 2: Plot only specific columns
print("\nOption 2: Plot specific feature columns")
print("Uncomment the lines below to execute:")
print("# output_folder = plot_geospatial_features(")
print("#     test_df, ")
print("#     output_folder='selected_feature_maps',")
print("#     include_cols=columns  # Use the 'columns' list defined above")
print("# )")

# Option 3: Plot features but exclude certain columns
print("\nOption 3: Plot with custom exclusions")
print("Uncomment the lines below to execute:")
print("# output_folder = plot_geospatial_features(")
print("#     test_df,")
print("#     output_folder='feature_maps',")
print("#     exclude_cols=['prospectivity', 'uncertainty']  # Add any custom exclusions")
print("# )")

print("\n" + "="*80)
print("To run: Uncomment one of the options above and execute this cell")
print("="*80)


In [ ]:
output_folder = plot_geospatial_features(
    test_df, 
    output_folder='selected_feature_maps',
    include_cols=columns  # Use the 'columns' list defined above
)
